# Project 2: Housing Price Prediction – Pass 22a

## Problem Definition


<!--
State the business problem. Translate the business problem into a Data Science problem by stating what kind of problem it is ( supervised vs unsupervised ) and whether it is a classification, regression, or clustering problem.
-->

Statement:

We want to predict housing sale prices based on various correlated feature data. This is a *supervised regression* problem, where the target of the prediction is sales prices. To accomplish this, we use a regression model alongside Regression prediction techniques, as well as feature scaling and regularization to minimize the RMS percentage error.

## Data Collection/Sources


Data Collection describes the process by which we obtain the data for use in the project. In this case, our data comes from S3 storage buckets on AWS, which we load into memory (variables) via URLs by passing them to the read_csv function of the pandas python library (more on that soon).

The data comes in one large set containing the full data available for use in the project. It is supplemented by a data dictionary that describes the set, the data types of the included data, and other information relevant to its use.

Finally, to store the data in a form we can work with in memory, we use a storage structure called a DataFrame object from the pandas python library. This object allows us to quickly view the data and preview operations on the data without changing it, as well as make updates to the data in a highly intuitive way.

Load Pandas, Numpy, and Matplotlib modules

In [903]:
# imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import statsmodels.api as sm

from sklearn import datasets, metrics
from sklearn import model_selection
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.metrics import make_scorer

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

from scipy import stats
from scipy import special

Data Triage:

In [904]:
%%bash

# In bash, save data URL into a bash variable 'url'
url="https://ddc-datascience.s3.amazonaws.com/Projects/Project.2-Housing/Data/Housing.Data.csv"
echo "${url}"

# Look at the CSV file header & display each field on its own line w/ line num
curl -s "${url}" | head -1 | tr , '\n' | cat -n

# Get the first 50 characters of the first 10 lines
{ time -p curl -s "${url}" | head -10 | cut -c1-50 | cat -n ; } 2>&1

# Fetch first 100 lines of csv file
curl -s "${url}" | head -c100 | file -

# Fetch header
curl -s -I "${url}"

https://ddc-datascience.s3.amazonaws.com/Projects/Project.2-Housing/Data/Housing.Data.csv
     1	PID
     2	MS SubClass
     3	MS Zoning
     4	Lot Frontage
     5	Lot Area
     6	Street
     7	Alley
     8	Lot Shape
     9	Land Contour
    10	Utilities
    11	Lot Config
    12	Land Slope
    13	Neighborhood
    14	Condition 1
    15	Condition 2
    16	Bldg Type
    17	House Style
    18	Overall Qual
    19	Overall Cond
    20	Year Built
    21	Year Remod/Add
    22	Roof Style
    23	Roof Matl
    24	Exterior 1st
    25	Exterior 2nd
    26	Mas Vnr Type
    27	Mas Vnr Area
    28	Exter Qual
    29	Exter Cond
    30	Foundation
    31	Bsmt Qual
    32	Bsmt Cond
    33	Bsmt Exposure
    34	BsmtFin Type 1
    35	BsmtFin SF 1
    36	BsmtFin Type 2
    37	BsmtFin SF 2
    38	Bsmt Unf SF
    39	Total Bsmt SF
    40	Heating
    41	Heating QC
    42	Central Air
    43	Electrical
    44	1st Flr SF
    45	2nd Flr SF
    46	Low Qual Fin SF
    47	Gr Liv Area
    48	Bsmt Full Bath
    49	Bsmt Half B

Now, Load data Housing.Data.csv from AWS S3:

In [905]:
# url
url = 'https://ddc-datascience.s3.amazonaws.com/Projects/Project.2-Housing/Data/Housing.Data.csv'

Next, make a preview DataFrame to look at columns:

In [906]:
# make preview
df_preview = pd.read_csv(url)
df_preview

,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,Utilities,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,905101070,20,RL,62.0,14299,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,MnPrv,NaN,0,7,2007,WD,Normal,115400
1,905101330,90,RL,72.0,10791,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,Shed,500,10,2006,WD,Normal,90000
2,903454090,50,RM,50.0,9000,Pave,NaN,Reg,Bnk,AllPub,...,0,NaN,NaN,NaN,0,12,2007,WD,Normal,141000
3,533244030,60,FV,68.0,7379,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,254000
4,909252020,70,RL,60.0,7200,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,MnPrv,NaN,0,4,2009,WD,Normal,155000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2632,903231070,50,RM,52.0,6240,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,MnPrv,Shed,400,9,2006,WD,Normal,114500
2633,906201021,80,RL,74.0,10778,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,7,2009,WD,Normal,162000
2634,533253070,120,RL,61.0,3782,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2009,WD,Normal,211500
2635,527376100,20,RL,78.0,10140,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,MnPrv,NaN,0,8,2009,WD,Normal,165000


In [907]:
# preview all column names
df_preview.columns

Index(['PID', 'MS SubClass', 'MS Zoning', 'Lot Frontage', 'Lot Area', 'Street',
       'Alley', 'Lot Shape', 'Land Contour', 'Utilities', 'Lot Config',
       'Land Slope', 'Neighborhood', 'Condition 1', 'Condition 2', 'Bldg Type',
       'House Style', 'Overall Qual', 'Overall Cond', 'Year Built',
       'Year Remod/Add', 'Roof Style', 'Roof Matl', 'Exterior 1st',
       'Exterior 2nd', 'Mas Vnr Type', 'Mas Vnr Area', 'Exter Qual',
       'Exter Cond', 'Foundation', 'Bsmt Qual', 'Bsmt Cond', 'Bsmt Exposure',
       'BsmtFin Type 1', 'BsmtFin SF 1', 'BsmtFin Type 2', 'BsmtFin SF 2',
       'Bsmt Unf SF', 'Total Bsmt SF', 'Heating', 'Heating QC', 'Central Air',
       'Electrical', '1st Flr SF', '2nd Flr SF', 'Low Qual Fin SF',
       'Gr Liv Area', 'Bsmt Full Bath', 'Bsmt Half Bath', 'Full Bath',
       'Half Bath', 'Bedroom AbvGr', 'Kitchen AbvGr', 'Kitchen Qual',
       'TotRms AbvGrd', 'Functional', 'Fireplaces', 'Fireplace Qu',
       'Garage Type', 'Garage Yr Blt', 'Garage Finish'

Define the target column (and features types, if required):

In [908]:
# define target column (by name) and features column(s)
target_col = "SalePrice"
feature_types = ['float64', 'int64', 'O']

Create list of target col and feature cols:

In [909]:
# use list comprehension to generate list of target col + selected feature cols
selected_cols = [
    col
    for col in df_preview.columns
    if (col == target_col) or (df_preview[col].dtype in feature_types)
]
selected_cols

['PID',
 'MS SubClass',
 'MS Zoning',
 'Lot Frontage',
 'Lot Area',
 'Street',
 'Alley',
 'Lot Shape',
 'Land Contour',
 'Utilities',
 'Lot Config',
 'Land Slope',
 'Neighborhood',
 'Condition 1',
 'Condition 2',
 'Bldg Type',
 'House Style',
 'Overall Qual',
 'Overall Cond',
 'Year Built',
 'Year Remod/Add',
 'Roof Style',
 'Roof Matl',
 'Exterior 1st',
 'Exterior 2nd',
 'Mas Vnr Type',
 'Mas Vnr Area',
 'Exter Qual',
 'Exter Cond',
 'Foundation',
 'Bsmt Qual',
 'Bsmt Cond',
 'Bsmt Exposure',
 'BsmtFin Type 1',
 'BsmtFin SF 1',
 'BsmtFin Type 2',
 'BsmtFin SF 2',
 'Bsmt Unf SF',
 'Total Bsmt SF',
 'Heating',
 'Heating QC',
 'Central Air',
 'Electrical',
 '1st Flr SF',
 '2nd Flr SF',
 'Low Qual Fin SF',
 'Gr Liv Area',
 'Bsmt Full Bath',
 'Bsmt Half Bath',
 'Full Bath',
 'Half Bath',
 'Bedroom AbvGr',
 'Kitchen AbvGr',
 'Kitchen Qual',
 'TotRms AbvGrd',
 'Functional',
 'Fireplaces',
 'Fireplace Qu',
 'Garage Type',
 'Garage Yr Blt',
 'Garage Finish',
 'Garage Cars',
 'Garage Area',
 'G

Read in the data from .csv using target_col and feature_types:

In [910]:
df_preview[selected_cols].dtypes.unique()

array([dtype('int64'), dtype('O'), dtype('float64')], dtype=object)

In [911]:
# make dataframe
df = pd.read_csv( url, usecols=selected_cols )

Finally, check the dataframe:

In [912]:
# check dataframe
df

,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,Utilities,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,905101070,20,RL,62.0,14299,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,MnPrv,NaN,0,7,2007,WD,Normal,115400
1,905101330,90,RL,72.0,10791,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,Shed,500,10,2006,WD,Normal,90000
2,903454090,50,RM,50.0,9000,Pave,NaN,Reg,Bnk,AllPub,...,0,NaN,NaN,NaN,0,12,2007,WD,Normal,141000
3,533244030,60,FV,68.0,7379,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,254000
4,909252020,70,RL,60.0,7200,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,MnPrv,NaN,0,4,2009,WD,Normal,155000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2632,903231070,50,RM,52.0,6240,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,MnPrv,Shed,400,9,2006,WD,Normal,114500
2633,906201021,80,RL,74.0,10778,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,7,2009,WD,Normal,162000
2634,533253070,120,RL,61.0,3782,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2009,WD,Normal,211500
2635,527376100,20,RL,78.0,10140,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,MnPrv,NaN,0,8,2009,WD,Normal,165000


## Data Cleaning


Data Cleaning is the process of preparing the data set for use in analysis. Many data sets contain bad or null (empty) values, which can cause errors when trying to work with the data and use it in code. Cleaning the data involves removing bad (null-containing) columns and/or rows from the set, as well as imputing (replacing via insertion) similar data into fields with missing or unusable values.

Here, to clean the data, we mostly use the pandas drop function included with pandas DataFrame objects, which allow us to drop columns and rows according to conditions we specify, much like SQL WHERE clauses.

### Data Cleaning — All

Examine the data using tools we have used in class. You should be identifying the various columns such as ...

    - target column
    - feature columns
    - identifier columns
    - null columns
    - null rows

    Tools you should be using include, but are not limited to:

    - head
    - tail
    - shape
    - size
    - info
    - describe
    - nunique
    - unique
    - isnull/isna



#### Data Exploration

First, define the target column:

In [913]:
# check target_col
target_col

'SalePrice'

- head

In [914]:
# head
df.head()

,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,Utilities,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,905101070,20,RL,62.0,14299,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,MnPrv,NaN,0,7,2007,WD,Normal,115400
1,905101330,90,RL,72.0,10791,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,Shed,500,10,2006,WD,Normal,90000
2,903454090,50,RM,50.0,9000,Pave,NaN,Reg,Bnk,AllPub,...,0,NaN,NaN,NaN,0,12,2007,WD,Normal,141000
3,533244030,60,FV,68.0,7379,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,254000
4,909252020,70,RL,60.0,7200,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,MnPrv,NaN,0,4,2009,WD,Normal,155000


- tail

In [915]:
# tail
df.tail()

,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,Utilities,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
2632,903231070,50,RM,52.0,6240,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,MnPrv,Shed,400,9,2006,WD,Normal,114500
2633,906201021,80,RL,74.0,10778,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,7,2009,WD,Normal,162000
2634,533253070,120,RL,61.0,3782,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2009,WD,Normal,211500
2635,527376100,20,RL,78.0,10140,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,MnPrv,NaN,0,8,2009,WD,Normal,165000
2636,528292060,60,RL,41.0,12460,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2008,WD,Normal,225000


- shape

In [916]:
# shape
df.shape

(2637, 81)

- size

In [917]:
# size
df.size

213597

- info

In [918]:
# info
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2637 entries, 0 to 2636
Data columns (total 81 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   PID              2637 non-null   int64  
 1   MS SubClass      2637 non-null   int64  
 2   MS Zoning        2637 non-null   object 
 3   Lot Frontage     2188 non-null   float64
 4   Lot Area         2637 non-null   int64  
 5   Street           2637 non-null   object 
 6   Alley            180 non-null    object 
 7   Lot Shape        2637 non-null   object 
 8   Land Contour     2637 non-null   object 
 9   Utilities        2637 non-null   object 
 10  Lot Config       2637 non-null   object 
 11  Land Slope       2637 non-null   object 
 12  Neighborhood     2637 non-null   object 
 13  Condition 1      2637 non-null   object 
 14  Condition 2      2637 non-null   object 
 15  Bldg Type        2637 non-null   object 
 16  House Style      2637 non-null   object 
 17  Overall Qual  

- describe

In [919]:
# describe
df.describe()

,PID,MS SubClass,Lot Frontage,Lot Area,Overall Qual,Overall Cond,Year Built,Year Remod/Add,Mas Vnr Area,BsmtFin SF 1,...,Wood Deck SF,Open Porch SF,Enclosed Porch,3Ssn Porch,Screen Porch,Pool Area,Misc Val,Mo Sold,Yr Sold,SalePrice
count,2.637000e+03,2637.000000,2188.000000,2637.000000,2637.000000,2637.000000,2637.000000,2637.000000,2614.000000,2636.000000,...,2637.000000,2637.000000,2637.000000,2637.000000,2637.000000,2637.000000,2637.000000,2637.000000,2637.000000,2637.000000
mean,7.141301e+08,57.349261,69.166819,10044.694729,6.097459,5.569966,1971.288586,1984.202882,101.887911,438.441199,...,94.305271,46.984452,22.813424,2.368601,15.775123,2.130072,42.014031,6.243838,2007.795601,179986.230186
std,1.887527e+08,42.499091,23.356779,6742.549521,1.411522,1.118262,30.306986,20.913077,179.578232,449.602326,...,126.927272,66.564333,61.177638,23.160600,55.783751,35.140140,393.158781,2.722093,1.306403,78309.251522
min,5.263011e+08,20.000000,21.000000,1300.000000,1.000000,1.000000,1872.000000,1950.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,2006.000000,12789.000000
25%,5.284770e+08,20.000000,58.000000,7436.000000,5.000000,5.000000,1954.000000,1965.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,4.000000,2007.000000,129500.000000
50%,5.354530e+08,50.000000,68.000000,9450.000000,6.000000,5.000000,1973.000000,1993.000000,0.000000,368.000000,...,0.000000,27.000000,0.000000,0.000000,0.000000,0.000000,0.000000,6.000000,2008.000000,160000.000000
75%,9.071870e+08,70.000000,80.000000,11526.000000,7.000000,6.000000,2001.000000,2004.000000,164.000000,732.000000,...,168.000000,70.000000,0.000000,0.000000,0.000000,0.000000,0.000000,8.000000,2009.000000,213000.000000
max,1.007100e+09,190.000000,313.000000,164660.000000,10.000000,9.000000,2010.000000,2010.000000,1600.000000,5644.000000,...,1424.000000,742.000000,584.000000,407.000000,576.000000,800.000000,12500.000000,12.000000,2010.000000,745000.000000


- nunique

In [920]:
# nunique
df.nunique().sort_values( ascending = False )

,0
PID,2637
Lot Area,1799
Gr Liv Area,1216
Bsmt Unf SF,1080
1st Flr SF,1022
...,...
Bsmt Half Bath,3
Paved Drive,3
Street,2
Alley,2


In [921]:
# normalize to look for ID column(s)
df.nunique().sort_values( ascending = False ) / df.shape[0] * 100

,0
PID,100.000000
Lot Area,68.221464
Gr Liv Area,46.113007
Bsmt Unf SF,40.955631
1st Flr SF,38.756162
...,...
Bsmt Half Bath,0.113766
Paved Drive,0.113766
Street,0.075844
Alley,0.075844


- unique

In [922]:
# unique
df[target_col].unique()

array([115400,  90000, 141000, 254000, 155000, 125000, 194500, 185000,
       235000, 152000, 201000, 207000, 192000, 174000, 190000, 176000,
       384500, 395039, 167000, 165000, 195000, 104500, 161000, 245700,
       226001, 153500, 139000,  35000, 181000, 212000, 156000, 139500,
       225000,  12789, 157500, 110000, 130000, 263000, 205000, 117000,
       200624, 120000, 137900, 245500, 107000, 327000,  93000, 145250,
       168000, 169900, 122000, 161750, 188000, 149500, 421250, 305000,
        50138, 136000, 150750, 307000, 241000,  86900, 202500,  93500,
       121000, 107500, 336000, 213000, 129000, 170000, 337500, 535000,
       173500, 162000,  37900, 158500,  94000, 350000, 140000, 409900,
       115000, 177000,  94750, 121600, 128500, 301000, 230000, 151000,
       137500,  85500, 129400, 191500, 250000, 135000, 119000, 150000,
       178000, 163900, 199000, 203000, 178900, 180000, 182000, 177900,
       122500, 134900, 132000, 147983,  68500, 158000, 260261,  64500,
      

- isnull

In [923]:
# isnull
df.isnull().sum().sort_values( ascending = False )

,0
Pool QC,2626
Misc Feature,2541
Alley,2457
Fence,2109
Mas Vnr Type,1607
...,...
Mo Sold,0
Yr Sold,0
Sale Type,0
Sale Condition,0


In [924]:
# percent null by column
(df.isnull().sum() / df.shape[0]).sort_values( ascending = False ) * 100

,0
Pool QC,99.582859
Misc Feature,96.359499
Alley,93.174061
Fence,79.977247
Mas Vnr Type,60.940463
...,...
Mo Sold,0.000000
Yr Sold,0.000000
Sale Type,0.000000
Sale Condition,0.000000


In [925]:
# look at unique values in the col-by-col null counts to classify cols
((df.isnull().sum() / df.shape[0]).sort_values( ascending = False ) * 100).unique()

array([9.95828593e+01, 9.63594994e+01, 9.31740614e+01, 7.99772469e+01,
       6.09404626e+01, 4.86916951e+01, 1.70269245e+01, 5.57451650e+00,
       5.49867273e+00, 2.88206295e+00, 2.80621919e+00, 2.76829731e+00,
       8.72203261e-01, 7.58437619e-02, 3.79218809e-02, 0.00000000e+00])

In [926]:
# check to see if anything is not zero -- should return 0 if no nulls
df.isnull().sum().sum()

np.int64(14208)

In [927]:
# how many rows with at least one null
num_rows_with_nulls = df.isna().any(axis=1).sum()
int(num_rows_with_nulls)

2637

The target (SalePrice) contains no nulls, but the feature (Lot Frontage) contains about 17% null values. To address this, we can impute values using the median value in that column.

- dtypes

In [928]:
# dtypes
df.dtypes

,0
PID,int64
MS SubClass,int64
MS Zoning,object
Lot Frontage,float64
Lot Area,int64
...,...
Mo Sold,int64
Yr Sold,int64
Sale Type,object
Sale Condition,object


Examine null columns, columns with some nulls, and columns with no nulls:

In [929]:
# display columns with all null values
df.isnull().sum()[df.isnull().sum() == df.shape[0]]

,0


In [930]:
# count these
int(df.isnull().sum()[df.isnull().sum() == df.shape[0]].count())

0

In [931]:
# show columns with some null values
df.isnull().sum()[(df.isnull().sum() != df.shape[0]) & (df.isnull().sum() != 0)].sort_values( ascending = False )

,0
Pool QC,2626
Misc Feature,2541
Alley,2457
Fence,2109
Mas Vnr Type,1607
Fireplace Qu,1284
Lot Frontage,449
Garage Cond,147
Garage Yr Blt,147
Garage Finish,147


In [932]:
# count these
int(df.isnull().sum()[(df.isnull().sum() != df.shape[0]) & (df.isnull().sum() != 0)].count())

26

In [933]:
# columns with no nulls
df.isnull().sum()[df.isnull().sum() == 0]

,0
PID,0
MS SubClass,0
MS Zoning,0
Lot Area,0
Street,0
Lot Shape,0
Land Contour,0
Utilities,0
Lot Config,0
Land Slope,0


In [934]:
# count these
int(df.isnull().sum()[df.isnull().sum() == 0].count())

55

- Target: Value Counts

In [935]:
# value counts on target
df[target_col].value_counts( dropna = False )

,count
SalePrice,
140000,31
135000,30
130000,28
155000,24
160000,22
...,...
160200,1
174190,1
415298,1


In [936]:
# value counts normalized
df[target_col].value_counts( dropna = False, normalize = True ) * 100

,proportion
SalePrice,
140000,1.175578
135000,1.137656
130000,1.061813
155000,0.910125
160000,0.834281
...,...
160200,0.037922
174190,0.037922
415298,0.037922


In [937]:
# target col size/shape
df[[target_col]].shape[0]

2637

In [938]:
# sum of nulls in target
int(df[[target_col]].isnull().sum().iloc[0])

0

In [939]:
# sum of non-nulls in target
df[[target_col]].shape[0] - int(df[[target_col]].isnull().sum().iloc[0])

2637

There are no nulls in the target column, which means no rows will need to be dropped on its account.

In [940]:
df.shape, df.size

((2637, 81), 213597)

In [941]:
df.drop(columns = [target_col]).columns

Index(['PID', 'MS SubClass', 'MS Zoning', 'Lot Frontage', 'Lot Area', 'Street',
       'Alley', 'Lot Shape', 'Land Contour', 'Utilities', 'Lot Config',
       'Land Slope', 'Neighborhood', 'Condition 1', 'Condition 2', 'Bldg Type',
       'House Style', 'Overall Qual', 'Overall Cond', 'Year Built',
       'Year Remod/Add', 'Roof Style', 'Roof Matl', 'Exterior 1st',
       'Exterior 2nd', 'Mas Vnr Type', 'Mas Vnr Area', 'Exter Qual',
       'Exter Cond', 'Foundation', 'Bsmt Qual', 'Bsmt Cond', 'Bsmt Exposure',
       'BsmtFin Type 1', 'BsmtFin SF 1', 'BsmtFin Type 2', 'BsmtFin SF 2',
       'Bsmt Unf SF', 'Total Bsmt SF', 'Heating', 'Heating QC', 'Central Air',
       'Electrical', '1st Flr SF', '2nd Flr SF', 'Low Qual Fin SF',
       'Gr Liv Area', 'Bsmt Full Bath', 'Bsmt Half Bath', 'Full Bath',
       'Half Bath', 'Bedroom AbvGr', 'Kitchen AbvGr', 'Kitchen Qual',
       'TotRms AbvGrd', 'Functional', 'Fireplaces', 'Fireplace Qu',
       'Garage Type', 'Garage Yr Blt', 'Garage Finish'

In [942]:
df.isna().any(axis=1).sum()

np.int64(2637)

In [943]:
df.columns

Index(['PID', 'MS SubClass', 'MS Zoning', 'Lot Frontage', 'Lot Area', 'Street',
       'Alley', 'Lot Shape', 'Land Contour', 'Utilities', 'Lot Config',
       'Land Slope', 'Neighborhood', 'Condition 1', 'Condition 2', 'Bldg Type',
       'House Style', 'Overall Qual', 'Overall Cond', 'Year Built',
       'Year Remod/Add', 'Roof Style', 'Roof Matl', 'Exterior 1st',
       'Exterior 2nd', 'Mas Vnr Type', 'Mas Vnr Area', 'Exter Qual',
       'Exter Cond', 'Foundation', 'Bsmt Qual', 'Bsmt Cond', 'Bsmt Exposure',
       'BsmtFin Type 1', 'BsmtFin SF 1', 'BsmtFin Type 2', 'BsmtFin SF 2',
       'Bsmt Unf SF', 'Total Bsmt SF', 'Heating', 'Heating QC', 'Central Air',
       'Electrical', '1st Flr SF', '2nd Flr SF', 'Low Qual Fin SF',
       'Gr Liv Area', 'Bsmt Full Bath', 'Bsmt Half Bath', 'Full Bath',
       'Half Bath', 'Bedroom AbvGr', 'Kitchen AbvGr', 'Kitchen Qual',
       'TotRms AbvGrd', 'Functional', 'Fireplaces', 'Fireplace Qu',
       'Garage Type', 'Garage Yr Blt', 'Garage Finish'

#### Exploration Summary

The dataset contains 2637 rows and 81 columns, making for a total of 213597 entries of data. Many of these fields are null.

Details:

  - target column: 'SalePrice'
  - feature columns: [
  'MS SubClass', 'MS Zoning', 'Lot Frontage', 'Lot Area', 'Street',
       'Alley', 'Lot Shape', 'Land Contour', 'Utilities', 'Lot Config',
       'Land Slope', 'Neighborhood', 'Condition 1', 'Condition 2', 'Bldg Type',
       'House Style', 'Overall Qual', 'Overall Cond', 'Year Built',
       'Year Remod/Add', 'Roof Style', 'Roof Matl', 'Exterior 1st',
       'Exterior 2nd', 'Mas Vnr Type', 'Mas Vnr Area', 'Exter Qual',
       'Exter Cond', 'Foundation', 'Bsmt Qual', 'Bsmt Cond', 'Bsmt Exposure',
       'BsmtFin Type 1', 'BsmtFin SF 1', 'BsmtFin Type 2', 'BsmtFin SF 2',
       'Bsmt Unf SF', 'Total Bsmt SF', 'Heating', 'Heating QC', 'Central Air',
       'Electrical', '1st Flr SF', '2nd Flr SF', 'Low Qual Fin SF',
       'Gr Liv Area', 'Bsmt Full Bath', 'Bsmt Half Bath', 'Full Bath',
       'Half Bath', 'Bedroom AbvGr', 'Kitchen AbvGr', 'Kitchen Qual',
       'TotRms AbvGrd', 'Functional', 'Fireplaces', 'Fireplace Qu',
       'Garage Type', 'Garage Yr Blt', 'Garage Finish', 'Garage Cars',
       'Garage Area', 'Garage Qual', 'Garage Cond', 'Paved Drive',
       'Wood Deck SF', 'Open Porch SF', 'Enclosed Porch', '3Ssn Porch',
       'Screen Porch', 'Pool Area', 'Pool QC', 'Fence', 'Misc Feature',
       'Misc Val', 'Mo Sold', 'Yr Sold', 'Sale Type', 'Sale Condition'
  ]
  - identifier columns: 'PID'
  - null columns: 0
  - null rows (target): 0
  - null rows (any): 2637 / 2637 (~100%)

### Partitioning Data

Next, we split our data into three groups by data type in order to properly clean each group according to its particular needs.

In [944]:
# declare a df_all DataFrame for all the data
df_all = df.copy()

In [945]:
# Split df into three separate data frames --> df_float
df_float = df_all.drop( columns = [target_col] ).select_dtypes( include = ['float64'] )
df_float = pd.concat([pd.DataFrame(df[target_col]), df_float], axis = 1)
df_float.head()

,SalePrice,Lot Frontage,Mas Vnr Area,BsmtFin SF 1,BsmtFin SF 2,Bsmt Unf SF,Total Bsmt SF,Bsmt Full Bath,Bsmt Half Bath,Garage Yr Blt,Garage Cars,Garage Area
0,115400,62.0,0.0,513.0,144.0,348.0,1005.0,1.0,0.0,1964.0,2.0,440.0
1,90000,72.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1967.0,2.0,516.0
2,141000,50.0,0.0,0.0,0.0,780.0,780.0,0.0,0.0,1979.0,1.0,544.0
3,254000,68.0,0.0,484.0,0.0,491.0,975.0,1.0,0.0,2000.0,2.0,592.0
4,155000,60.0,0.0,350.0,210.0,0.0,560.0,1.0,0.0,1971.0,2.0,576.0


In [946]:
# Split df into three separate data frames --> df_int
df_int = df_all.drop( columns = [target_col] ).select_dtypes( include = ['int64'] )
df_int = pd.concat([pd.DataFrame(df[target_col]), df_int], axis = 1)
df_int.head()

,SalePrice,PID,MS SubClass,Lot Area,Overall Qual,Overall Cond,Year Built,Year Remod/Add,1st Flr SF,2nd Flr SF,...,Fireplaces,Wood Deck SF,Open Porch SF,Enclosed Porch,3Ssn Porch,Screen Porch,Pool Area,Misc Val,Mo Sold,Yr Sold
0,115400,905101070,20,14299,4,3,1964,1964,1005,0,...,0,0,0,0,0,0,0,0,7,2007
1,90000,905101330,90,10791,4,5,1967,1967,1296,0,...,0,0,0,0,0,0,0,500,10,2006
2,141000,903454090,50,9000,6,6,1937,1950,780,595,...,1,0,162,0,0,126,0,0,12,2007
3,254000,533244030,60,7379,8,5,2000,2000,975,873,...,1,280,184,0,0,0,0,0,4,2010
4,155000,909252020,70,7200,7,9,1936,2007,575,560,...,0,256,0,0,0,0,0,0,4,2009


In [947]:
# Split df into three separate data frames --> df_object
df_object = df_all.drop( columns = [target_col] ).select_dtypes( include = ['O'] )
df_object = pd.concat([pd.DataFrame(df[target_col]), df_object], axis = 1)
df_object.head()

,SalePrice,MS Zoning,Street,Alley,Lot Shape,Land Contour,Utilities,Lot Config,Land Slope,Neighborhood,...,Garage Type,Garage Finish,Garage Qual,Garage Cond,Paved Drive,Pool QC,Fence,Misc Feature,Sale Type,Sale Condition
0,115400,RL,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,Sawyer,...,Detchd,Unf,TA,TA,N,NaN,MnPrv,NaN,WD,Normal
1,90000,RL,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,Sawyer,...,CarPort,Unf,TA,TA,Y,NaN,NaN,Shed,WD,Normal
2,141000,RM,Pave,NaN,Reg,Bnk,AllPub,Inside,Gtl,IDOTRR,...,Detchd,Unf,TA,TA,P,NaN,NaN,NaN,WD,Normal
3,254000,FV,Pave,NaN,IR1,Lvl,AllPub,Inside,Gtl,Somerst,...,Attchd,RFn,TA,TA,Y,NaN,NaN,NaN,WD,Normal
4,155000,RL,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,SWISU,...,Detchd,RFn,TA,TA,Y,NaN,MnPrv,NaN,WD,Normal


### Data Cleaning — Floats

#### Data Exploration — Floats

In [948]:
# start by initializing df to df_float
df = df_float.copy()

- head

In [949]:
# head
df.head()

,SalePrice,Lot Frontage,Mas Vnr Area,BsmtFin SF 1,BsmtFin SF 2,Bsmt Unf SF,Total Bsmt SF,Bsmt Full Bath,Bsmt Half Bath,Garage Yr Blt,Garage Cars,Garage Area
0,115400,62.0,0.0,513.0,144.0,348.0,1005.0,1.0,0.0,1964.0,2.0,440.0
1,90000,72.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1967.0,2.0,516.0
2,141000,50.0,0.0,0.0,0.0,780.0,780.0,0.0,0.0,1979.0,1.0,544.0
3,254000,68.0,0.0,484.0,0.0,491.0,975.0,1.0,0.0,2000.0,2.0,592.0
4,155000,60.0,0.0,350.0,210.0,0.0,560.0,1.0,0.0,1971.0,2.0,576.0


- tail

In [950]:
# tail
df.tail()

,SalePrice,Lot Frontage,Mas Vnr Area,BsmtFin SF 1,BsmtFin SF 2,Bsmt Unf SF,Total Bsmt SF,Bsmt Full Bath,Bsmt Half Bath,Garage Yr Blt,Garage Cars,Garage Area
2632,114500,52.0,0.0,0.0,0.0,816.0,816.0,0.0,0.0,1985.0,2.0,528.0
2633,162000,74.0,0.0,460.0,286.0,308.0,1054.0,1.0,0.0,1990.0,2.0,462.0
2634,211500,61.0,0.0,960.0,0.0,266.0,1226.0,1.0,0.0,1981.0,2.0,484.0
2635,165000,78.0,174.0,0.0,0.0,1064.0,1064.0,0.0,0.0,1974.0,2.0,478.0
2636,225000,41.0,0.0,870.0,0.0,167.0,1037.0,0.0,0.0,1999.0,2.0,400.0


- shape

In [951]:
# shape
df.shape

(2637, 12)

- size

In [952]:
# size
df.size

31644

- info

In [953]:
# info
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2637 entries, 0 to 2636
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   SalePrice       2637 non-null   int64  
 1   Lot Frontage    2188 non-null   float64
 2   Mas Vnr Area    2614 non-null   float64
 3   BsmtFin SF 1    2636 non-null   float64
 4   BsmtFin SF 2    2636 non-null   float64
 5   Bsmt Unf SF     2636 non-null   float64
 6   Total Bsmt SF   2636 non-null   float64
 7   Bsmt Full Bath  2635 non-null   float64
 8   Bsmt Half Bath  2635 non-null   float64
 9   Garage Yr Blt   2490 non-null   float64
 10  Garage Cars     2636 non-null   float64
 11  Garage Area     2636 non-null   float64
dtypes: float64(11), int64(1)
memory usage: 247.3 KB


- describe

In [954]:
# describe
df.describe()

,SalePrice,Lot Frontage,Mas Vnr Area,BsmtFin SF 1,BsmtFin SF 2,Bsmt Unf SF,Total Bsmt SF,Bsmt Full Bath,Bsmt Half Bath,Garage Yr Blt,Garage Cars,Garage Area
count,2637.000000,2188.000000,2614.000000,2636.000000,2636.000000,2636.000000,2636.000000,2635.000000,2635.000000,2490.000000,2636.000000,2636.000000
mean,179986.230186,69.166819,101.887911,438.441199,51.068665,555.669196,1045.179059,0.432258,0.058824,1978.185944,1.763278,470.968134
std,78309.251522,23.356779,179.578232,449.602326,172.521803,436.223188,429.686089,0.525239,0.240130,25.452647,0.760576,212.816905
min,12789.000000,21.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1895.000000,0.000000,0.000000
25%,129500.000000,58.000000,0.000000,0.000000,0.000000,218.000000,792.000000,0.000000,0.000000,1961.000000,1.000000,324.750000
50%,160000.000000,68.000000,0.000000,368.000000,0.000000,464.500000,988.000000,0.000000,0.000000,1979.000000,2.000000,480.000000
75%,213000.000000,80.000000,164.000000,732.000000,0.000000,797.250000,1290.250000,1.000000,0.000000,2002.000000,2.000000,576.000000
max,745000.000000,313.000000,1600.000000,5644.000000,1526.000000,2336.000000,6110.000000,3.000000,2.000000,2207.000000,5.000000,1488.000000


- nunique

In [955]:
# nunique
df.nunique().sort_values( ascending = False )

,0
Bsmt Unf SF,1080
Total Bsmt SF,1001
SalePrice,983
BsmtFin SF 1,935
Garage Area,576
Mas Vnr Area,425
BsmtFin SF 2,253
Lot Frontage,126
Garage Yr Blt,102
Garage Cars,6


In [956]:
# normalize to look for ID column(s)
df.nunique().sort_values( ascending = False ) / df.shape[0] * 100

,0
Bsmt Unf SF,40.955631
Total Bsmt SF,37.959803
SalePrice,37.277209
BsmtFin SF 1,35.456959
Garage Area,21.843003
Mas Vnr Area,16.116799
BsmtFin SF 2,9.594236
Lot Frontage,4.778157
Garage Yr Blt,3.868032
Garage Cars,0.227531


- unique

In [957]:
# unique
df[target_col].unique()

array([115400,  90000, 141000, 254000, 155000, 125000, 194500, 185000,
       235000, 152000, 201000, 207000, 192000, 174000, 190000, 176000,
       384500, 395039, 167000, 165000, 195000, 104500, 161000, 245700,
       226001, 153500, 139000,  35000, 181000, 212000, 156000, 139500,
       225000,  12789, 157500, 110000, 130000, 263000, 205000, 117000,
       200624, 120000, 137900, 245500, 107000, 327000,  93000, 145250,
       168000, 169900, 122000, 161750, 188000, 149500, 421250, 305000,
        50138, 136000, 150750, 307000, 241000,  86900, 202500,  93500,
       121000, 107500, 336000, 213000, 129000, 170000, 337500, 535000,
       173500, 162000,  37900, 158500,  94000, 350000, 140000, 409900,
       115000, 177000,  94750, 121600, 128500, 301000, 230000, 151000,
       137500,  85500, 129400, 191500, 250000, 135000, 119000, 150000,
       178000, 163900, 199000, 203000, 178900, 180000, 182000, 177900,
       122500, 134900, 132000, 147983,  68500, 158000, 260261,  64500,
      

- isnull

In [958]:
# isnull
df.isnull().sum().sort_values( ascending = False )

,0
Lot Frontage,449
Garage Yr Blt,147
Mas Vnr Area,23
Bsmt Full Bath,2
Bsmt Half Bath,2
BsmtFin SF 1,1
Total Bsmt SF,1
BsmtFin SF 2,1
Garage Cars,1
Bsmt Unf SF,1


In [959]:
# percent null by column
(df.isnull().sum() / df.shape[0]).sort_values( ascending = False ) * 100

,0
Lot Frontage,17.026925
Garage Yr Blt,5.574516
Mas Vnr Area,0.872203
Bsmt Full Bath,0.075844
Bsmt Half Bath,0.075844
BsmtFin SF 1,0.037922
Total Bsmt SF,0.037922
BsmtFin SF 2,0.037922
Garage Cars,0.037922
Bsmt Unf SF,0.037922


In [960]:
# look at unique values in the col-by-col null counts to classify cols
((df.isnull().sum() / df.shape[0]).sort_values( ascending = False ) * 100).unique()

array([17.02692454,  5.5745165 ,  0.87220326,  0.07584376,  0.03792188,
        0.        ])

In [961]:
# check to see if anything is not zero -- should return 0 if no nulls
df.isnull().sum().sum()

np.int64(629)

In [962]:
# how many rows with at least one null
num_rows_with_nulls = df.isna().any(axis=1).sum()
int(num_rows_with_nulls)

604

Many different null counts exist throughout the data set, indicating that this will not be a straightforward nulls cleanup.

- dtypes

In [963]:
# dtypes
df.dtypes

,0
SalePrice,int64
Lot Frontage,float64
Mas Vnr Area,float64
BsmtFin SF 1,float64
BsmtFin SF 2,float64
Bsmt Unf SF,float64
Total Bsmt SF,float64
Bsmt Full Bath,float64
Bsmt Half Bath,float64
Garage Yr Blt,float64


Examine null columns, columns with some nulls, and columns with no nulls:

In [964]:
# display columns with all null values
df.isnull().sum()[df.isnull().sum() == df.shape[0]]

,0


In [965]:
# count these
int(df.isnull().sum()[df.isnull().sum() == df.shape[0]].count())

0

In [966]:
# show columns with some null values
df.isnull().sum()[(df.isnull().sum() != df.shape[0]) & (df.isnull().sum() != 0)].sort_values( ascending = False )

,0
Lot Frontage,449
Garage Yr Blt,147
Mas Vnr Area,23
Bsmt Half Bath,2
Bsmt Full Bath,2
BsmtFin SF 1,1
BsmtFin SF 2,1
Total Bsmt SF,1
Bsmt Unf SF,1
Garage Cars,1


In [967]:
# count these
int(df.isnull().sum()[(df.isnull().sum() != df.shape[0]) & (df.isnull().sum() != 0)].count())

11

In [968]:
# columns with no nulls
df.isnull().sum()[df.isnull().sum() == 0]

,0
SalePrice,0


In [969]:
# count these
int(df.isnull().sum()[df.isnull().sum() == 0].count())

1

- Target: Value Counts

In [970]:
# value counts on target
df[target_col].value_counts( dropna = False )

,count
SalePrice,
140000,31
135000,30
130000,28
155000,24
160000,22
...,...
160200,1
174190,1
415298,1


In [971]:
# value counts normalized
df[target_col].value_counts( dropna = False, normalize = True ) * 100

,proportion
SalePrice,
140000,1.175578
135000,1.137656
130000,1.061813
155000,0.910125
160000,0.834281
...,...
160200,0.037922
174190,0.037922
415298,0.037922


In [972]:
# target col size/shape
df[[target_col]].shape[0]

2637

In [973]:
# sum of nulls in target
int(df[[target_col]].isnull().sum().iloc[0])

0

In [974]:
# sum of non-nulls in target
df[[target_col]].shape[0] - int(df[[target_col]].isnull().sum().iloc[0])

2637

There are no nulls in the target column, which means no rows will need to be dropped on its account.

#### Exploration Summary — Floats

The floats+ints-feature dataset contains 2637 rows and 38 columns, making for a total of 100206 entries of data. Many of these fields are null.

Details:

  - target column: 'SalePrice'
  - feature columns: [
       'Lot Frontage', 'Garage Yr Blt', 'Mas Vnr Area', 'Bsmt Full Bath',
       'Bsmt Half Bath', 'BsmtFin SF 2', 'BsmtFin SF 1', 'Total Bsmt SF',
       'Bsmt Unf SF', 'Garage Area', 'Garage Cars',
       'MS SubClass', 'Lot Area', 'Overall Qual', 'Overall Cond', 'Year Built', 'Year Remod/Add', '1st Flr SF', '2nd Flr SF', 'Low Qual Fin SF', 'Gr Liv Area', 'Full Bath', 'Half Bath', 'Bedroom AbvGr', 'Kitchen AbvGr', 'TotRms AbvGrd', 'Fireplaces', 'Wood Deck SF', 'Open Porch SF', 'Enclosed Porch', '3Ssn Porch', 'Screen Porch', 'Pool Area', 'Misc Val', 'Mo Sold', 'Yr Sold'
       ]
  - identifier columns: 'PID'
  - null columns: 0
  - null rows (target): 0
  - null rows (any): 2637 (100%)

#### Cleanup — Floats

If there are data cleaning issues, develop recommendations for how to deal with them.  Specifically, handle these common issues:

    - identifer columns
    - target column with null counts
    - feature columns with high null counts
    - feature columns with low null counts
    - feature columns with medium null counts


First, make a backup:

In [975]:
# make backup
df_f_bak = df.copy()

In [976]:
# restore from backup
df = df_f_bak.copy()

In [977]:
# df again to double-check
df

,SalePrice,Lot Frontage,Mas Vnr Area,BsmtFin SF 1,BsmtFin SF 2,Bsmt Unf SF,Total Bsmt SF,Bsmt Full Bath,Bsmt Half Bath,Garage Yr Blt,Garage Cars,Garage Area
0,115400,62.0,0.0,513.0,144.0,348.0,1005.0,1.0,0.0,1964.0,2.0,440.0
1,90000,72.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1967.0,2.0,516.0
2,141000,50.0,0.0,0.0,0.0,780.0,780.0,0.0,0.0,1979.0,1.0,544.0
3,254000,68.0,0.0,484.0,0.0,491.0,975.0,1.0,0.0,2000.0,2.0,592.0
4,155000,60.0,0.0,350.0,210.0,0.0,560.0,1.0,0.0,1971.0,2.0,576.0
...,...,...,...,...,...,...,...,...,...,...,...,...
2632,114500,52.0,0.0,0.0,0.0,816.0,816.0,0.0,0.0,1985.0,2.0,528.0
2633,162000,74.0,0.0,460.0,286.0,308.0,1054.0,1.0,0.0,1990.0,2.0,462.0
2634,211500,61.0,0.0,960.0,0.0,266.0,1226.0,1.0,0.0,1981.0,2.0,484.0
2635,165000,78.0,174.0,0.0,0.0,1064.0,1064.0,0.0,0.0,1974.0,2.0,478.0


1. Remove identifier columns

- No identifier columns

In [978]:
# backup 1
df_f_bak1 = df.copy()

In [979]:
# restore from backup 1
df = df_f_bak1.copy()

In [980]:
df

,SalePrice,Lot Frontage,Mas Vnr Area,BsmtFin SF 1,BsmtFin SF 2,Bsmt Unf SF,Total Bsmt SF,Bsmt Full Bath,Bsmt Half Bath,Garage Yr Blt,Garage Cars,Garage Area
0,115400,62.0,0.0,513.0,144.0,348.0,1005.0,1.0,0.0,1964.0,2.0,440.0
1,90000,72.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1967.0,2.0,516.0
2,141000,50.0,0.0,0.0,0.0,780.0,780.0,0.0,0.0,1979.0,1.0,544.0
3,254000,68.0,0.0,484.0,0.0,491.0,975.0,1.0,0.0,2000.0,2.0,592.0
4,155000,60.0,0.0,350.0,210.0,0.0,560.0,1.0,0.0,1971.0,2.0,576.0
...,...,...,...,...,...,...,...,...,...,...,...,...
2632,114500,52.0,0.0,0.0,0.0,816.0,816.0,0.0,0.0,1985.0,2.0,528.0
2633,162000,74.0,0.0,460.0,286.0,308.0,1054.0,1.0,0.0,1990.0,2.0,462.0
2634,211500,61.0,0.0,960.0,0.0,266.0,1226.0,1.0,0.0,1981.0,2.0,484.0
2635,165000,78.0,174.0,0.0,0.0,1064.0,1064.0,0.0,0.0,1974.0,2.0,478.0


2. Drop null target rows



- No null target rows

3. Drop feature columns with many or all nulls

In [981]:
df.isnull().sum().sort_values( ascending = False )

,0
Lot Frontage,449
Garage Yr Blt,147
Mas Vnr Area,23
Bsmt Full Bath,2
Bsmt Half Bath,2
BsmtFin SF 1,1
Total Bsmt SF,1
BsmtFin SF 2,1
Garage Cars,1
Bsmt Unf SF,1


In [982]:
# df = df.loc[:, df.isnull().mean() <= 0.5]
df = df.drop( columns = [index for index in list(df.isnull().sum().index) if (df.isnull().sum().loc[index] / df.shape[0] > 0.4)])

In [983]:
df

,SalePrice,Lot Frontage,Mas Vnr Area,BsmtFin SF 1,BsmtFin SF 2,Bsmt Unf SF,Total Bsmt SF,Bsmt Full Bath,Bsmt Half Bath,Garage Yr Blt,Garage Cars,Garage Area
0,115400,62.0,0.0,513.0,144.0,348.0,1005.0,1.0,0.0,1964.0,2.0,440.0
1,90000,72.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1967.0,2.0,516.0
2,141000,50.0,0.0,0.0,0.0,780.0,780.0,0.0,0.0,1979.0,1.0,544.0
3,254000,68.0,0.0,484.0,0.0,491.0,975.0,1.0,0.0,2000.0,2.0,592.0
4,155000,60.0,0.0,350.0,210.0,0.0,560.0,1.0,0.0,1971.0,2.0,576.0
...,...,...,...,...,...,...,...,...,...,...,...,...
2632,114500,52.0,0.0,0.0,0.0,816.0,816.0,0.0,0.0,1985.0,2.0,528.0
2633,162000,74.0,0.0,460.0,286.0,308.0,1054.0,1.0,0.0,1990.0,2.0,462.0
2634,211500,61.0,0.0,960.0,0.0,266.0,1226.0,1.0,0.0,1981.0,2.0,484.0
2635,165000,78.0,174.0,0.0,0.0,1064.0,1064.0,0.0,0.0,1974.0,2.0,478.0


In [984]:
list(df.isnull().sum().index)

['SalePrice',
 'Lot Frontage',
 'Mas Vnr Area',
 'BsmtFin SF 1',
 'BsmtFin SF 2',
 'Bsmt Unf SF',
 'Total Bsmt SF',
 'Bsmt Full Bath',
 'Bsmt Half Bath',
 'Garage Yr Blt',
 'Garage Cars',
 'Garage Area']

In [985]:
df.shape

(2637, 12)

4. Drop remaining records with null counts (if few)

In [986]:
# backup 2
df_f_bak2 = df.copy()

In [987]:
# restore from backup 2
df = df_f_bak2.copy()

In [988]:
df

,SalePrice,Lot Frontage,Mas Vnr Area,BsmtFin SF 1,BsmtFin SF 2,Bsmt Unf SF,Total Bsmt SF,Bsmt Full Bath,Bsmt Half Bath,Garage Yr Blt,Garage Cars,Garage Area
0,115400,62.0,0.0,513.0,144.0,348.0,1005.0,1.0,0.0,1964.0,2.0,440.0
1,90000,72.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1967.0,2.0,516.0
2,141000,50.0,0.0,0.0,0.0,780.0,780.0,0.0,0.0,1979.0,1.0,544.0
3,254000,68.0,0.0,484.0,0.0,491.0,975.0,1.0,0.0,2000.0,2.0,592.0
4,155000,60.0,0.0,350.0,210.0,0.0,560.0,1.0,0.0,1971.0,2.0,576.0
...,...,...,...,...,...,...,...,...,...,...,...,...
2632,114500,52.0,0.0,0.0,0.0,816.0,816.0,0.0,0.0,1985.0,2.0,528.0
2633,162000,74.0,0.0,460.0,286.0,308.0,1054.0,1.0,0.0,1990.0,2.0,462.0
2634,211500,61.0,0.0,960.0,0.0,266.0,1226.0,1.0,0.0,1981.0,2.0,484.0
2635,165000,78.0,174.0,0.0,0.0,1064.0,1064.0,0.0,0.0,1974.0,2.0,478.0


In [989]:
df.isnull().sum().sort_values( ascending = False )

,0
Lot Frontage,449
Garage Yr Blt,147
Mas Vnr Area,23
Bsmt Full Bath,2
Bsmt Half Bath,2
BsmtFin SF 1,1
Total Bsmt SF,1
BsmtFin SF 2,1
Garage Cars,1
Bsmt Unf SF,1


In [990]:
df = df.dropna(subset=[index for index in list(df.isnull().sum().index) if (df.isnull().sum().loc[index] / df.shape[0] < 0.05)])

In [991]:
df

,SalePrice,Lot Frontage,Mas Vnr Area,BsmtFin SF 1,BsmtFin SF 2,Bsmt Unf SF,Total Bsmt SF,Bsmt Full Bath,Bsmt Half Bath,Garage Yr Blt,Garage Cars,Garage Area
0,115400,62.0,0.0,513.0,144.0,348.0,1005.0,1.0,0.0,1964.0,2.0,440.0
1,90000,72.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1967.0,2.0,516.0
2,141000,50.0,0.0,0.0,0.0,780.0,780.0,0.0,0.0,1979.0,1.0,544.0
3,254000,68.0,0.0,484.0,0.0,491.0,975.0,1.0,0.0,2000.0,2.0,592.0
4,155000,60.0,0.0,350.0,210.0,0.0,560.0,1.0,0.0,1971.0,2.0,576.0
...,...,...,...,...,...,...,...,...,...,...,...,...
2632,114500,52.0,0.0,0.0,0.0,816.0,816.0,0.0,0.0,1985.0,2.0,528.0
2633,162000,74.0,0.0,460.0,286.0,308.0,1054.0,1.0,0.0,1990.0,2.0,462.0
2634,211500,61.0,0.0,960.0,0.0,266.0,1226.0,1.0,0.0,1981.0,2.0,484.0
2635,165000,78.0,174.0,0.0,0.0,1064.0,1064.0,0.0,0.0,1974.0,2.0,478.0


In [992]:
list(df.isnull().sum().index)

['SalePrice',
 'Lot Frontage',
 'Mas Vnr Area',
 'BsmtFin SF 1',
 'BsmtFin SF 2',
 'Bsmt Unf SF',
 'Total Bsmt SF',
 'Bsmt Full Bath',
 'Bsmt Half Bath',
 'Garage Yr Blt',
 'Garage Cars',
 'Garage Area']

In [993]:
df.shape

(2611, 12)

5.  impute with the median for remaining fields

In [994]:
# backup 3
df_f_bak3 = df.copy()

In [995]:
# restore from backup 3
df = df_f_bak3.copy()

In [996]:
df

,SalePrice,Lot Frontage,Mas Vnr Area,BsmtFin SF 1,BsmtFin SF 2,Bsmt Unf SF,Total Bsmt SF,Bsmt Full Bath,Bsmt Half Bath,Garage Yr Blt,Garage Cars,Garage Area
0,115400,62.0,0.0,513.0,144.0,348.0,1005.0,1.0,0.0,1964.0,2.0,440.0
1,90000,72.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1967.0,2.0,516.0
2,141000,50.0,0.0,0.0,0.0,780.0,780.0,0.0,0.0,1979.0,1.0,544.0
3,254000,68.0,0.0,484.0,0.0,491.0,975.0,1.0,0.0,2000.0,2.0,592.0
4,155000,60.0,0.0,350.0,210.0,0.0,560.0,1.0,0.0,1971.0,2.0,576.0
...,...,...,...,...,...,...,...,...,...,...,...,...
2632,114500,52.0,0.0,0.0,0.0,816.0,816.0,0.0,0.0,1985.0,2.0,528.0
2633,162000,74.0,0.0,460.0,286.0,308.0,1054.0,1.0,0.0,1990.0,2.0,462.0
2634,211500,61.0,0.0,960.0,0.0,266.0,1226.0,1.0,0.0,1981.0,2.0,484.0
2635,165000,78.0,174.0,0.0,0.0,1064.0,1064.0,0.0,0.0,1974.0,2.0,478.0


In [997]:
# check again how many of each column is null
df.isnull().sum().sort_values( ascending = False )

,0
Lot Frontage,443
Garage Yr Blt,145
Mas Vnr Area,0
SalePrice,0
BsmtFin SF 1,0
BsmtFin SF 2,0
Total Bsmt SF,0
Bsmt Unf SF,0
Bsmt Full Bath,0
Bsmt Half Bath,0


In [998]:
# normalize null counts (per column)
df.isnull().sum().sort_values( ascending = False ) / df.shape[0] * 100

,0
Lot Frontage,16.966679
Garage Yr Blt,5.553428
Mas Vnr Area,0.000000
SalePrice,0.000000
BsmtFin SF 1,0.000000
BsmtFin SF 2,0.000000
Total Bsmt SF,0.000000
Bsmt Unf SF,0.000000
Bsmt Full Bath,0.000000
Bsmt Half Bath,0.000000


Impute:

In [999]:
feature_cols = list(df.drop(columns = target_col).columns)
feature_cols

['Lot Frontage',
 'Mas Vnr Area',
 'BsmtFin SF 1',
 'BsmtFin SF 2',
 'Bsmt Unf SF',
 'Total Bsmt SF',
 'Bsmt Full Bath',
 'Bsmt Half Bath',
 'Garage Yr Blt',
 'Garage Cars',
 'Garage Area']

In [1000]:
# Define categorical float cols (need the mode, not the mean)
categorical_float_cols = ['Bsmt Full Bath', 'Bsmt Half Bath', 'Garage Yr Built' 'Garage Cars']

In [1001]:
# loop through features to bulk impute with respective column medians
for feature_col in feature_cols:

  if feature_col not in categorical_float_cols:

    # Calculate the median
    median = df[feature_col].median()
    print(median)

    # Assign median to nulls
    df.fillna( {feature_col: median}, inplace = True )

    # Verify median is about the same as before
    print(df[feature_col].median())

  else:

    # Calculate the mode
    mode = df[feature_col].mode(dropna = True)[0]
    print(mode)

    # Assign median to nulls
    df.fillna( {feature_col: mode}, inplace = True )

    # Verify median is about the same as before
    print(df[feature_col].mode()[0])

68.0
68.0
0.0
0.0
368.0
368.0
0.0
0.0
463.0
463.0
988.0
988.0
0.0
0.0
0.0
0.0
1979.0
1979.0
2.0
2.0
478.0
478.0


In [1002]:
df

,SalePrice,Lot Frontage,Mas Vnr Area,BsmtFin SF 1,BsmtFin SF 2,Bsmt Unf SF,Total Bsmt SF,Bsmt Full Bath,Bsmt Half Bath,Garage Yr Blt,Garage Cars,Garage Area
0,115400,62.0,0.0,513.0,144.0,348.0,1005.0,1.0,0.0,1964.0,2.0,440.0
1,90000,72.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1967.0,2.0,516.0
2,141000,50.0,0.0,0.0,0.0,780.0,780.0,0.0,0.0,1979.0,1.0,544.0
3,254000,68.0,0.0,484.0,0.0,491.0,975.0,1.0,0.0,2000.0,2.0,592.0
4,155000,60.0,0.0,350.0,210.0,0.0,560.0,1.0,0.0,1971.0,2.0,576.0
...,...,...,...,...,...,...,...,...,...,...,...,...
2632,114500,52.0,0.0,0.0,0.0,816.0,816.0,0.0,0.0,1985.0,2.0,528.0
2633,162000,74.0,0.0,460.0,286.0,308.0,1054.0,1.0,0.0,1990.0,2.0,462.0
2634,211500,61.0,0.0,960.0,0.0,266.0,1226.0,1.0,0.0,1981.0,2.0,484.0
2635,165000,78.0,174.0,0.0,0.0,1064.0,1064.0,0.0,0.0,1974.0,2.0,478.0


In [1003]:
# check again how many of each column is null
df.isnull().sum().sort_values( ascending = False )

,0
SalePrice,0
Lot Frontage,0
Mas Vnr Area,0
BsmtFin SF 1,0
BsmtFin SF 2,0
Bsmt Unf SF,0
Total Bsmt SF,0
Bsmt Full Bath,0
Bsmt Half Bath,0
Garage Yr Blt,0


In [1004]:
# backup 4
df_f_bak4 = df.copy()

In [1005]:
# restore from backup 4
df = df_f_bak4.copy()

In [1006]:
df

,SalePrice,Lot Frontage,Mas Vnr Area,BsmtFin SF 1,BsmtFin SF 2,Bsmt Unf SF,Total Bsmt SF,Bsmt Full Bath,Bsmt Half Bath,Garage Yr Blt,Garage Cars,Garage Area
0,115400,62.0,0.0,513.0,144.0,348.0,1005.0,1.0,0.0,1964.0,2.0,440.0
1,90000,72.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1967.0,2.0,516.0
2,141000,50.0,0.0,0.0,0.0,780.0,780.0,0.0,0.0,1979.0,1.0,544.0
3,254000,68.0,0.0,484.0,0.0,491.0,975.0,1.0,0.0,2000.0,2.0,592.0
4,155000,60.0,0.0,350.0,210.0,0.0,560.0,1.0,0.0,1971.0,2.0,576.0
...,...,...,...,...,...,...,...,...,...,...,...,...
2632,114500,52.0,0.0,0.0,0.0,816.0,816.0,0.0,0.0,1985.0,2.0,528.0
2633,162000,74.0,0.0,460.0,286.0,308.0,1054.0,1.0,0.0,1990.0,2.0,462.0
2634,211500,61.0,0.0,960.0,0.0,266.0,1226.0,1.0,0.0,1981.0,2.0,484.0
2635,165000,78.0,174.0,0.0,0.0,1064.0,1064.0,0.0,0.0,1974.0,2.0,478.0


In [1007]:
# check total null counts sum
f"Number of remaining nulls in df: {int(df.isnull().sum().sum())}"

'Number of remaining nulls in df: 0'

In [1008]:
# backup
df_float_bak = df_float.copy()

In [1009]:
# restore
df_float = df_float_bak.copy()

In [1010]:
# update
df_float = df.copy()

### Data Cleaning — Ints

#### Data Exploration — Ints

In [1011]:
# start by initializing df to df_int
df = df_int.copy()

- head

In [1012]:
# head
df.head()

,SalePrice,PID,MS SubClass,Lot Area,Overall Qual,Overall Cond,Year Built,Year Remod/Add,1st Flr SF,2nd Flr SF,...,Fireplaces,Wood Deck SF,Open Porch SF,Enclosed Porch,3Ssn Porch,Screen Porch,Pool Area,Misc Val,Mo Sold,Yr Sold
0,115400,905101070,20,14299,4,3,1964,1964,1005,0,...,0,0,0,0,0,0,0,0,7,2007
1,90000,905101330,90,10791,4,5,1967,1967,1296,0,...,0,0,0,0,0,0,0,500,10,2006
2,141000,903454090,50,9000,6,6,1937,1950,780,595,...,1,0,162,0,0,126,0,0,12,2007
3,254000,533244030,60,7379,8,5,2000,2000,975,873,...,1,280,184,0,0,0,0,0,4,2010
4,155000,909252020,70,7200,7,9,1936,2007,575,560,...,0,256,0,0,0,0,0,0,4,2009


- tail

In [1013]:
# tail
df.tail()

,SalePrice,PID,MS SubClass,Lot Area,Overall Qual,Overall Cond,Year Built,Year Remod/Add,1st Flr SF,2nd Flr SF,...,Fireplaces,Wood Deck SF,Open Porch SF,Enclosed Porch,3Ssn Porch,Screen Porch,Pool Area,Misc Val,Mo Sold,Yr Sold
2632,114500,903231070,50,6240,6,6,1934,1950,816,0,...,1,112,0,0,0,0,0,400,9,2006
2633,162000,906201021,80,10778,7,6,1990,1991,1061,0,...,0,114,36,0,0,0,0,0,7,2009
2634,211500,533253070,120,3782,8,5,1981,1981,1226,0,...,2,133,78,0,0,0,0,0,9,2009
2635,165000,527376100,20,10140,6,5,1974,1974,1350,0,...,1,0,0,0,0,0,0,0,8,2009
2636,225000,528292060,60,12460,7,5,1999,2000,1037,1285,...,1,144,44,0,0,0,0,0,5,2008


- shape

In [1014]:
# shape
df.shape

(2637, 27)

- size

In [1015]:
# size
df.size

71199

- info

In [1016]:
# info
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2637 entries, 0 to 2636
Data columns (total 27 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   SalePrice        2637 non-null   int64
 1   PID              2637 non-null   int64
 2   MS SubClass      2637 non-null   int64
 3   Lot Area         2637 non-null   int64
 4   Overall Qual     2637 non-null   int64
 5   Overall Cond     2637 non-null   int64
 6   Year Built       2637 non-null   int64
 7   Year Remod/Add   2637 non-null   int64
 8   1st Flr SF       2637 non-null   int64
 9   2nd Flr SF       2637 non-null   int64
 10  Low Qual Fin SF  2637 non-null   int64
 11  Gr Liv Area      2637 non-null   int64
 12  Full Bath        2637 non-null   int64
 13  Half Bath        2637 non-null   int64
 14  Bedroom AbvGr    2637 non-null   int64
 15  Kitchen AbvGr    2637 non-null   int64
 16  TotRms AbvGrd    2637 non-null   int64
 17  Fireplaces       2637 non-null   int64
 18  Wood Dec

- describe

In [1017]:
# describe
df.describe()

,SalePrice,PID,MS SubClass,Lot Area,Overall Qual,Overall Cond,Year Built,Year Remod/Add,1st Flr SF,2nd Flr SF,...,Fireplaces,Wood Deck SF,Open Porch SF,Enclosed Porch,3Ssn Porch,Screen Porch,Pool Area,Misc Val,Mo Sold,Yr Sold
count,2637.000000,2.637000e+03,2637.000000,2637.000000,2637.000000,2637.000000,2637.000000,2637.000000,2637.000000,2637.000000,...,2637.000000,2637.000000,2637.000000,2637.000000,2637.000000,2637.000000,2637.000000,2637.000000,2637.000000,2637.000000
mean,179986.230186,7.141301e+08,57.349261,10044.694729,6.097459,5.569966,1971.288586,1984.202882,1155.505878,337.004930,...,0.599166,94.305271,46.984452,22.813424,2.368601,15.775123,2.130072,42.014031,6.243838,2007.795601
std,78309.251522,1.887527e+08,42.499091,6742.549521,1.411522,1.118262,30.306986,20.913077,382.597499,427.742219,...,0.651052,126.927272,66.564333,61.177638,23.160600,55.783751,35.140140,393.158781,2.722093,1.306403
min,12789.000000,5.263011e+08,20.000000,1300.000000,1.000000,1.000000,1872.000000,1950.000000,334.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,2006.000000
25%,129500.000000,5.284770e+08,20.000000,7436.000000,5.000000,5.000000,1954.000000,1965.000000,878.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,4.000000,2007.000000
50%,160000.000000,5.354530e+08,50.000000,9450.000000,6.000000,5.000000,1973.000000,1993.000000,1082.000000,0.000000,...,1.000000,0.000000,27.000000,0.000000,0.000000,0.000000,0.000000,0.000000,6.000000,2008.000000
75%,213000.000000,9.071870e+08,70.000000,11526.000000,7.000000,6.000000,2001.000000,2004.000000,1380.000000,708.000000,...,1.000000,168.000000,70.000000,0.000000,0.000000,0.000000,0.000000,0.000000,8.000000,2009.000000
max,745000.000000,1.007100e+09,190.000000,164660.000000,10.000000,9.000000,2010.000000,2010.000000,4692.000000,2065.000000,...,4.000000,1424.000000,742.000000,584.000000,407.000000,576.000000,800.000000,12500.000000,12.000000,2010.000000


- nunique

In [1018]:
# nunique
df.nunique().sort_values( ascending = False )

,0
PID,2637
Lot Area,1799
Gr Liv Area,1216
1st Flr SF,1022
SalePrice,983
2nd Flr SF,589
Wood Deck SF,367
Open Porch SF,238
Enclosed Porch,175
Year Built,118


In [1019]:
# normalize to look for ID column(s)
df.nunique().sort_values( ascending = False ) / df.shape[0] * 100

,0
PID,100.000000
Lot Area,68.221464
Gr Liv Area,46.113007
1st Flr SF,38.756162
SalePrice,37.277209
2nd Flr SF,22.335988
Wood Deck SF,13.917330
Open Porch SF,9.025408
Enclosed Porch,6.636329
Year Built,4.474782


- unique

In [1020]:
# unique
df[target_col].unique()

array([115400,  90000, 141000, 254000, 155000, 125000, 194500, 185000,
       235000, 152000, 201000, 207000, 192000, 174000, 190000, 176000,
       384500, 395039, 167000, 165000, 195000, 104500, 161000, 245700,
       226001, 153500, 139000,  35000, 181000, 212000, 156000, 139500,
       225000,  12789, 157500, 110000, 130000, 263000, 205000, 117000,
       200624, 120000, 137900, 245500, 107000, 327000,  93000, 145250,
       168000, 169900, 122000, 161750, 188000, 149500, 421250, 305000,
        50138, 136000, 150750, 307000, 241000,  86900, 202500,  93500,
       121000, 107500, 336000, 213000, 129000, 170000, 337500, 535000,
       173500, 162000,  37900, 158500,  94000, 350000, 140000, 409900,
       115000, 177000,  94750, 121600, 128500, 301000, 230000, 151000,
       137500,  85500, 129400, 191500, 250000, 135000, 119000, 150000,
       178000, 163900, 199000, 203000, 178900, 180000, 182000, 177900,
       122500, 134900, 132000, 147983,  68500, 158000, 260261,  64500,
      

- isnull

In [1021]:
# isnull
df.isnull().sum().sort_values( ascending = False )

,0
SalePrice,0
PID,0
MS SubClass,0
Lot Area,0
Overall Qual,0
Overall Cond,0
Year Built,0
Year Remod/Add,0
1st Flr SF,0
2nd Flr SF,0


In [1022]:
# percent null by column
(df.isnull().sum() / df.shape[0]).sort_values( ascending = False ) * 100

,0
SalePrice,0.0
PID,0.0
MS SubClass,0.0
Lot Area,0.0
Overall Qual,0.0
Overall Cond,0.0
Year Built,0.0
Year Remod/Add,0.0
1st Flr SF,0.0
2nd Flr SF,0.0


In [1023]:
# look at unique values in the col-by-col null counts to classify cols
((df.isnull().sum() / df.shape[0]).sort_values( ascending = False ) * 100).unique()

array([0.])

In [1024]:
# check to see if anything is not zero -- should return 0 if no nulls
df.isnull().sum().sum()

np.int64(0)

In [1025]:
# how many rows with at least one null
num_rows_with_nulls = df.isna().any(axis=1).sum()
int(num_rows_with_nulls)

0

Many different null counts exist throughout the data set, indicating that this will not be a straightforward nulls cleanup.

- dtypes

In [1026]:
# dtypes
df.dtypes

,0
SalePrice,int64
PID,int64
MS SubClass,int64
Lot Area,int64
Overall Qual,int64
Overall Cond,int64
Year Built,int64
Year Remod/Add,int64
1st Flr SF,int64
2nd Flr SF,int64


Examine null columns, columns with some nulls, and columns with no nulls:

In [1027]:
# display columns with all null values
df.isnull().sum()[df.isnull().sum() == df.shape[0]]

,0


In [1028]:
# count these
int(df.isnull().sum()[df.isnull().sum() == df.shape[0]].count())

0

In [1029]:
# show columns with some null values
df.isnull().sum()[(df.isnull().sum() != df.shape[0]) & (df.isnull().sum() != 0)].sort_values( ascending = False )

,0


In [1030]:
# count these
int(df.isnull().sum()[(df.isnull().sum() != df.shape[0]) & (df.isnull().sum() != 0)].count())

0

In [1031]:
# columns with no nulls
df.isnull().sum()[df.isnull().sum() == 0]

,0
SalePrice,0
PID,0
MS SubClass,0
Lot Area,0
Overall Qual,0
Overall Cond,0
Year Built,0
Year Remod/Add,0
1st Flr SF,0
2nd Flr SF,0


In [1032]:
# count these
int(df.isnull().sum()[df.isnull().sum() == 0].count())

27

- Target: Value Counts

In [1033]:
# value counts on target
df[target_col].value_counts( dropna = False )

,count
SalePrice,
140000,31
135000,30
130000,28
155000,24
160000,22
...,...
160200,1
174190,1
415298,1


In [1034]:
# value counts normalized
df[target_col].value_counts( dropna = False, normalize = True ) * 100

,proportion
SalePrice,
140000,1.175578
135000,1.137656
130000,1.061813
155000,0.910125
160000,0.834281
...,...
160200,0.037922
174190,0.037922
415298,0.037922


In [1035]:
# target col size/shape
df[[target_col]].shape[0]

2637

In [1036]:
# sum of nulls in target
int(df[[target_col]].isnull().sum().iloc[0])

0

In [1037]:
# sum of non-nulls in target
df[[target_col]].shape[0] - int(df[[target_col]].isnull().sum().iloc[0])

2637

There are no nulls in the target column, which means no rows will need to be dropped on its account.

#### Exploration Summary — Ints

The ints-feature dataset contains 2637 rows and 27 columns, making for a total of 71199 entries of data. Conveniently, none of these fields is null; as such, no columns or rows need to be removed, and no values need to be imputed.

Details:

  - target column: 'SalePrice'
  - feature columns: ['PID', 'MS SubClass', 'Lot Area', 'Overall Qual', 'Overall Cond',
       'Year Built', 'Year Remod/Add', '1st Flr SF', '2nd Flr SF',
       'Low Qual Fin SF', 'Gr Liv Area', 'Full Bath', 'Half Bath',
       'Bedroom AbvGr', 'Kitchen AbvGr', 'TotRms AbvGrd', 'Fireplaces',
       'Wood Deck SF', 'Open Porch SF', 'Enclosed Porch', '3Ssn Porch',
       'Screen Porch', 'Pool Area', 'Misc Val', 'Mo Sold', 'Yr Sold']
  - identifier columns: 'PID'
  - null columns: 0
  - null rows (target): 0
  - null rows (any): 0 (0%)

#### Cleanup — Ints

If there are data cleaning issues, develop recommendations for how to deal with them.  Specifically, handle these common issues:

    - identifer columns
    - target column with null counts
    - feature columns with high null counts
    - feature columns with low null counts
    - feature columns with medium null counts


First, make a backup:

In [1038]:
# make backup
df_i_bak = df.copy()

In [1039]:
# restore from backup
df = df_i_bak.copy()

In [1040]:
#df again to double-check
df

,SalePrice,PID,MS SubClass,Lot Area,Overall Qual,Overall Cond,Year Built,Year Remod/Add,1st Flr SF,2nd Flr SF,...,Fireplaces,Wood Deck SF,Open Porch SF,Enclosed Porch,3Ssn Porch,Screen Porch,Pool Area,Misc Val,Mo Sold,Yr Sold
0,115400,905101070,20,14299,4,3,1964,1964,1005,0,...,0,0,0,0,0,0,0,0,7,2007
1,90000,905101330,90,10791,4,5,1967,1967,1296,0,...,0,0,0,0,0,0,0,500,10,2006
2,141000,903454090,50,9000,6,6,1937,1950,780,595,...,1,0,162,0,0,126,0,0,12,2007
3,254000,533244030,60,7379,8,5,2000,2000,975,873,...,1,280,184,0,0,0,0,0,4,2010
4,155000,909252020,70,7200,7,9,1936,2007,575,560,...,0,256,0,0,0,0,0,0,4,2009
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2632,114500,903231070,50,6240,6,6,1934,1950,816,0,...,1,112,0,0,0,0,0,400,9,2006
2633,162000,906201021,80,10778,7,6,1990,1991,1061,0,...,0,114,36,0,0,0,0,0,7,2009
2634,211500,533253070,120,3782,8,5,1981,1981,1226,0,...,2,133,78,0,0,0,0,0,9,2009
2635,165000,527376100,20,10140,6,5,1974,1974,1350,0,...,1,0,0,0,0,0,0,0,8,2009


1. Remove identifier columns

In [1041]:
# remove identifier columns from DataFrame
df = df.drop( columns = [ 'PID' ] )

In [1042]:
# backup 2
df_i_bak2 = df.copy()

In [1043]:
# restore from backup 2
df = df_i_bak2.copy()

In [1044]:
df

,SalePrice,MS SubClass,Lot Area,Overall Qual,Overall Cond,Year Built,Year Remod/Add,1st Flr SF,2nd Flr SF,Low Qual Fin SF,...,Fireplaces,Wood Deck SF,Open Porch SF,Enclosed Porch,3Ssn Porch,Screen Porch,Pool Area,Misc Val,Mo Sold,Yr Sold
0,115400,20,14299,4,3,1964,1964,1005,0,0,...,0,0,0,0,0,0,0,0,7,2007
1,90000,90,10791,4,5,1967,1967,1296,0,0,...,0,0,0,0,0,0,0,500,10,2006
2,141000,50,9000,6,6,1937,1950,780,595,0,...,1,0,162,0,0,126,0,0,12,2007
3,254000,60,7379,8,5,2000,2000,975,873,0,...,1,280,184,0,0,0,0,0,4,2010
4,155000,70,7200,7,9,1936,2007,575,560,0,...,0,256,0,0,0,0,0,0,4,2009
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2632,114500,50,6240,6,6,1934,1950,816,0,360,...,1,112,0,0,0,0,0,400,9,2006
2633,162000,80,10778,7,6,1990,1991,1061,0,0,...,0,114,36,0,0,0,0,0,7,2009
2634,211500,120,3782,8,5,1981,1981,1226,0,0,...,2,133,78,0,0,0,0,0,9,2009
2635,165000,20,10140,6,5,1974,1974,1350,0,0,...,1,0,0,0,0,0,0,0,8,2009


2. Drop null target rows



- No null target rows

3. Drop feature columns with many or all nulls

- No columns with many or all nulls

4. Drop remaining records with null counts (if few)

- No remaining rows with null counts

5.  impute with the median for remaining fields

- No remaining fields are empty, so no imputations necessary

In [1045]:
# check again how many of each column is null
df.isnull().sum().sort_values( ascending = False )

,0
SalePrice,0
MS SubClass,0
Lot Area,0
Overall Qual,0
Overall Cond,0
Year Built,0
Year Remod/Add,0
1st Flr SF,0
2nd Flr SF,0
Low Qual Fin SF,0


In [1046]:
# check total null counts sum
f"Number of remaining nulls in df: {int(df.isnull().sum().sum())}"

'Number of remaining nulls in df: 0'

In [1047]:
# backup
df_int_bak = df_int.copy()

In [1048]:
# restore
df_int = df_int_bak.copy()

In [1049]:
# update
df_int = df.copy()

### Data Cleaning — Objects

#### Data Exploration — Objects

In [1050]:
# start by initializing df to df_object
df = df_object.copy()

- head

In [1051]:
# head
df.head()

,SalePrice,MS Zoning,Street,Alley,Lot Shape,Land Contour,Utilities,Lot Config,Land Slope,Neighborhood,...,Garage Type,Garage Finish,Garage Qual,Garage Cond,Paved Drive,Pool QC,Fence,Misc Feature,Sale Type,Sale Condition
0,115400,RL,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,Sawyer,...,Detchd,Unf,TA,TA,N,NaN,MnPrv,NaN,WD,Normal
1,90000,RL,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,Sawyer,...,CarPort,Unf,TA,TA,Y,NaN,NaN,Shed,WD,Normal
2,141000,RM,Pave,NaN,Reg,Bnk,AllPub,Inside,Gtl,IDOTRR,...,Detchd,Unf,TA,TA,P,NaN,NaN,NaN,WD,Normal
3,254000,FV,Pave,NaN,IR1,Lvl,AllPub,Inside,Gtl,Somerst,...,Attchd,RFn,TA,TA,Y,NaN,NaN,NaN,WD,Normal
4,155000,RL,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,SWISU,...,Detchd,RFn,TA,TA,Y,NaN,MnPrv,NaN,WD,Normal


- tail

In [1052]:
# tail
df.tail()

,SalePrice,MS Zoning,Street,Alley,Lot Shape,Land Contour,Utilities,Lot Config,Land Slope,Neighborhood,...,Garage Type,Garage Finish,Garage Qual,Garage Cond,Paved Drive,Pool QC,Fence,Misc Feature,Sale Type,Sale Condition
2632,114500,RM,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,BrkSide,...,Detchd,Unf,TA,TA,Y,NaN,MnPrv,Shed,WD,Normal
2633,162000,RL,Pave,NaN,IR1,Lvl,AllPub,Inside,Gtl,SawyerW,...,Attchd,Unf,TA,TA,Y,NaN,NaN,NaN,WD,Normal
2634,211500,RL,Pave,NaN,IR1,Lvl,AllPub,CulDSac,Gtl,Greens,...,Attchd,Fin,TA,TA,Y,NaN,NaN,NaN,WD,Normal
2635,165000,RL,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,NWAmes,...,Attchd,RFn,TA,TA,Y,NaN,MnPrv,NaN,WD,Normal
2636,225000,RL,Pave,NaN,IR1,Lvl,AllPub,FR2,Gtl,Gilbert,...,BuiltIn,Fin,TA,TA,Y,NaN,NaN,NaN,WD,Normal


- shape

In [1053]:
# shape
df.shape

(2637, 44)

- size

In [1054]:
# size
df.size

116028

- info

In [1055]:
# info
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2637 entries, 0 to 2636
Data columns (total 44 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   SalePrice       2637 non-null   int64 
 1   MS Zoning       2637 non-null   object
 2   Street          2637 non-null   object
 3   Alley           180 non-null    object
 4   Lot Shape       2637 non-null   object
 5   Land Contour    2637 non-null   object
 6   Utilities       2637 non-null   object
 7   Lot Config      2637 non-null   object
 8   Land Slope      2637 non-null   object
 9   Neighborhood    2637 non-null   object
 10  Condition 1     2637 non-null   object
 11  Condition 2     2637 non-null   object
 12  Bldg Type       2637 non-null   object
 13  House Style     2637 non-null   object
 14  Roof Style      2637 non-null   object
 15  Roof Matl       2637 non-null   object
 16  Exterior 1st    2637 non-null   object
 17  Exterior 2nd    2637 non-null   object
 18  Mas Vnr 

- describe

In [1056]:
# describe
df.describe()

,SalePrice
count,2637.000000
mean,179986.230186
std,78309.251522
min,12789.000000
25%,129500.000000
50%,160000.000000
75%,213000.000000
max,745000.000000


- nunique

In [1057]:
# nunique
df.nunique().sort_values( ascending = False )

,0
SalePrice,983
Neighborhood,28
Exterior 2nd,17
Exterior 1st,16
Sale Type,10
Condition 1,9
House Style,8
Functional,8
Roof Matl,8
Condition 2,8


In [1058]:
# normalize to look for ID column(s)
df.nunique().sort_values( ascending = False ) / df.shape[0] * 100

,0
SalePrice,37.277209
Neighborhood,1.061813
Exterior 2nd,0.644672
Exterior 1st,0.606750
Sale Type,0.379219
Condition 1,0.341297
House Style,0.303375
Functional,0.303375
Roof Matl,0.303375
Condition 2,0.303375


- unique

In [1059]:
# unique
df[target_col].unique()

array([115400,  90000, 141000, 254000, 155000, 125000, 194500, 185000,
       235000, 152000, 201000, 207000, 192000, 174000, 190000, 176000,
       384500, 395039, 167000, 165000, 195000, 104500, 161000, 245700,
       226001, 153500, 139000,  35000, 181000, 212000, 156000, 139500,
       225000,  12789, 157500, 110000, 130000, 263000, 205000, 117000,
       200624, 120000, 137900, 245500, 107000, 327000,  93000, 145250,
       168000, 169900, 122000, 161750, 188000, 149500, 421250, 305000,
        50138, 136000, 150750, 307000, 241000,  86900, 202500,  93500,
       121000, 107500, 336000, 213000, 129000, 170000, 337500, 535000,
       173500, 162000,  37900, 158500,  94000, 350000, 140000, 409900,
       115000, 177000,  94750, 121600, 128500, 301000, 230000, 151000,
       137500,  85500, 129400, 191500, 250000, 135000, 119000, 150000,
       178000, 163900, 199000, 203000, 178900, 180000, 182000, 177900,
       122500, 134900, 132000, 147983,  68500, 158000, 260261,  64500,
      

- isnull

In [1060]:
# isnull
df.isnull().sum().sort_values( ascending = False )

,0
Pool QC,2626
Misc Feature,2541
Alley,2457
Fence,2109
Mas Vnr Type,1607
Fireplace Qu,1284
Garage Cond,147
Garage Finish,147
Garage Qual,147
Garage Type,145


In [1061]:
# percent null by column
(df.isnull().sum() / df.shape[0]).sort_values( ascending = False ) * 100

,0
Pool QC,99.582859
Misc Feature,96.359499
Alley,93.174061
Fence,79.977247
Mas Vnr Type,60.940463
Fireplace Qu,48.691695
Garage Cond,5.574516
Garage Finish,5.574516
Garage Qual,5.574516
Garage Type,5.498673


In [1062]:
# look at unique values in the col-by-col null counts to classify cols
((df.isnull().sum() / df.shape[0]).sort_values( ascending = False ) * 100).unique()

array([99.58285931, 96.35949943, 93.17406143, 79.97724687, 60.94046265,
       48.69169511,  5.5745165 ,  5.49867273,  2.88206295,  2.80621919,
        2.76829731,  0.        ])

In [1063]:
# check to see if anything is not zero -- should return 0 if no nulls
df.isnull().sum().sum()

np.int64(13579)

In [1064]:
# how many rows with at least one null
num_rows_with_nulls = df.isna().any(axis=1).sum()
int(num_rows_with_nulls)

2637

The target (SalePrice) contains no nulls, but the feature (Lot Frontage) contains about 17% null values. To address this, we can impute values using the median value in that column.

- dtypes

In [1065]:
# dtypes
df.dtypes

,0
SalePrice,int64
MS Zoning,object
Street,object
Alley,object
Lot Shape,object
Land Contour,object
Utilities,object
Lot Config,object
Land Slope,object
Neighborhood,object


Examine null columns, columns with some nulls, and columns with no nulls:

In [1066]:
# display columns with all null values
df.isnull().sum()[df.isnull().sum() == df.shape[0]]

,0


In [1067]:
# count these
int(df.isnull().sum()[df.isnull().sum() == df.shape[0]].count())

0

In [1068]:
# show columns with some null values
df.isnull().sum()[(df.isnull().sum() != df.shape[0]) & (df.isnull().sum() != 0)].sort_values( ascending = False )

,0
Pool QC,2626
Misc Feature,2541
Alley,2457
Fence,2109
Mas Vnr Type,1607
Fireplace Qu,1284
Garage Finish,147
Garage Cond,147
Garage Qual,147
Garage Type,145


In [1069]:
# count these
int(df.isnull().sum()[(df.isnull().sum() != df.shape[0]) & (df.isnull().sum() != 0)].count())

15

In [1070]:
# columns with no nulls
df.isnull().sum()[df.isnull().sum() == 0]

,0
SalePrice,0
MS Zoning,0
Street,0
Lot Shape,0
Land Contour,0
Utilities,0
Lot Config,0
Land Slope,0
Neighborhood,0
Condition 1,0


In [1071]:
# count these
int(df.isnull().sum()[df.isnull().sum() == 0].count())

29

- Target: Value Counts

In [1072]:
# value counts on target
df[target_col].value_counts( dropna = False )

,count
SalePrice,
140000,31
135000,30
130000,28
155000,24
160000,22
...,...
160200,1
174190,1
415298,1


In [1073]:
# value counts normalized
df[target_col].value_counts( dropna = False, normalize = True ) * 100

,proportion
SalePrice,
140000,1.175578
135000,1.137656
130000,1.061813
155000,0.910125
160000,0.834281
...,...
160200,0.037922
174190,0.037922
415298,0.037922


In [1074]:
# target col size/shape
df[[target_col]].shape[0]

2637

In [1075]:
# sum of nulls in target
int(df[[target_col]].isnull().sum().iloc[0])

0

In [1076]:
# sum of non-nulls in target
df[[target_col]].shape[0] - int(df[[target_col]].isnull().sum().iloc[0])

2637

There are no nulls in the target column, which means no rows will need to be dropped on its account.

In [1077]:
df.shape, df.size

((2637, 44), 116028)

In [1078]:
df.drop(columns = [target_col]).columns

Index(['MS Zoning', 'Street', 'Alley', 'Lot Shape', 'Land Contour',
       'Utilities', 'Lot Config', 'Land Slope', 'Neighborhood', 'Condition 1',
       'Condition 2', 'Bldg Type', 'House Style', 'Roof Style', 'Roof Matl',
       'Exterior 1st', 'Exterior 2nd', 'Mas Vnr Type', 'Exter Qual',
       'Exter Cond', 'Foundation', 'Bsmt Qual', 'Bsmt Cond', 'Bsmt Exposure',
       'BsmtFin Type 1', 'BsmtFin Type 2', 'Heating', 'Heating QC',
       'Central Air', 'Electrical', 'Kitchen Qual', 'Functional',
       'Fireplace Qu', 'Garage Type', 'Garage Finish', 'Garage Qual',
       'Garage Cond', 'Paved Drive', 'Pool QC', 'Fence', 'Misc Feature',
       'Sale Type', 'Sale Condition'],
      dtype='object')

In [1079]:
df.isna().any(axis=1).sum()

np.int64(2637)

#### Exploration Summary — Objects

This abridged dataset contains 2637 rows and 44 columns, making for a total of 116028 entries of data. Many of these fields are null.

Details:

  - target column: 'SalePrice'
  - feature columns: [
  'MS Zoning', 'Street', 'Alley', 'Lot Shape', 'Land Contour',
       'Utilities', 'Lot Config', 'Land Slope', 'Neighborhood', 'Condition 1',
       'Condition 2', 'Bldg Type', 'House Style', 'Roof Style', 'Roof Matl',
       'Exterior 1st', 'Exterior 2nd', 'Mas Vnr Type', 'Exter Qual',
       'Exter Cond', 'Foundation', 'Bsmt Qual', 'Bsmt Cond', 'Bsmt Exposure',
       'BsmtFin Type 1', 'BsmtFin Type 2', 'Heating', 'Heating QC',
       'Central Air', 'Electrical', 'Kitchen Qual', 'Functional',
       'Fireplace Qu', 'Garage Type', 'Garage Finish', 'Garage Qual',
       'Garage Cond', 'Paved Drive', 'Pool QC', 'Fence', 'Misc Feature',
       'Sale Type', 'Sale Condition'
  ]
  - identifier columns: n/a
  - null columns: 0
  - null rows (target): 0
  - null rows (any): 2637 / 2637 (~100%)

#### Cleanup — Objects

If there are data cleaning issues, develop recommendations for how to deal with them.  Specifically, handle these common issues:

    - identifer columns
    - target column with null counts
    - feature columns with high null counts
    - feature columns with low null counts
    - feature columns with medium null counts


In [1080]:
# head again to double-check
df.head()

,SalePrice,MS Zoning,Street,Alley,Lot Shape,Land Contour,Utilities,Lot Config,Land Slope,Neighborhood,...,Garage Type,Garage Finish,Garage Qual,Garage Cond,Paved Drive,Pool QC,Fence,Misc Feature,Sale Type,Sale Condition
0,115400,RL,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,Sawyer,...,Detchd,Unf,TA,TA,N,NaN,MnPrv,NaN,WD,Normal
1,90000,RL,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,Sawyer,...,CarPort,Unf,TA,TA,Y,NaN,NaN,Shed,WD,Normal
2,141000,RM,Pave,NaN,Reg,Bnk,AllPub,Inside,Gtl,IDOTRR,...,Detchd,Unf,TA,TA,P,NaN,NaN,NaN,WD,Normal
3,254000,FV,Pave,NaN,IR1,Lvl,AllPub,Inside,Gtl,Somerst,...,Attchd,RFn,TA,TA,Y,NaN,NaN,NaN,WD,Normal
4,155000,RL,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,SWISU,...,Detchd,RFn,TA,TA,Y,NaN,MnPrv,NaN,WD,Normal


In [1081]:
# make backup
df_o_bak = df.copy()

In [1082]:
# restore from backup
df = df_o_bak.copy()

In [1083]:
df

,SalePrice,MS Zoning,Street,Alley,Lot Shape,Land Contour,Utilities,Lot Config,Land Slope,Neighborhood,...,Garage Type,Garage Finish,Garage Qual,Garage Cond,Paved Drive,Pool QC,Fence,Misc Feature,Sale Type,Sale Condition
0,115400,RL,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,Sawyer,...,Detchd,Unf,TA,TA,N,NaN,MnPrv,NaN,WD,Normal
1,90000,RL,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,Sawyer,...,CarPort,Unf,TA,TA,Y,NaN,NaN,Shed,WD,Normal
2,141000,RM,Pave,NaN,Reg,Bnk,AllPub,Inside,Gtl,IDOTRR,...,Detchd,Unf,TA,TA,P,NaN,NaN,NaN,WD,Normal
3,254000,FV,Pave,NaN,IR1,Lvl,AllPub,Inside,Gtl,Somerst,...,Attchd,RFn,TA,TA,Y,NaN,NaN,NaN,WD,Normal
4,155000,RL,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,SWISU,...,Detchd,RFn,TA,TA,Y,NaN,MnPrv,NaN,WD,Normal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2632,114500,RM,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,BrkSide,...,Detchd,Unf,TA,TA,Y,NaN,MnPrv,Shed,WD,Normal
2633,162000,RL,Pave,NaN,IR1,Lvl,AllPub,Inside,Gtl,SawyerW,...,Attchd,Unf,TA,TA,Y,NaN,NaN,NaN,WD,Normal
2634,211500,RL,Pave,NaN,IR1,Lvl,AllPub,CulDSac,Gtl,Greens,...,Attchd,Fin,TA,TA,Y,NaN,NaN,NaN,WD,Normal
2635,165000,RL,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,NWAmes,...,Attchd,RFn,TA,TA,Y,NaN,MnPrv,NaN,WD,Normal


In [1084]:
# Setup drop thresholds

# Drop a column if null % above 85%
drop_column_threshold = 0.85

# Drop a row if null % below 15%
drop_row_threshold = 0.15

1. Remove identifier columns

- No identifier rows present in set (culled at read level)

2. Drop null target rows



- No null target rows

3. Drop feature columns with many or all nulls

In [1085]:
# Select columns where the missing value ratio is less than drop_column_threshold
high_nulls = df.columns[df.isnull().mean() > drop_column_threshold]
high_nulls

Index(['Alley', 'Pool QC', 'Misc Feature'], dtype='object')

In [1086]:
# Drop those specific columns
df = df.drop(columns=high_nulls)
df

,SalePrice,MS Zoning,Street,Lot Shape,Land Contour,Utilities,Lot Config,Land Slope,Neighborhood,Condition 1,...,Functional,Fireplace Qu,Garage Type,Garage Finish,Garage Qual,Garage Cond,Paved Drive,Fence,Sale Type,Sale Condition
0,115400,RL,Pave,Reg,Lvl,AllPub,Inside,Gtl,Sawyer,Feedr,...,Typ,NaN,Detchd,Unf,TA,TA,N,MnPrv,WD,Normal
1,90000,RL,Pave,Reg,Lvl,AllPub,Inside,Gtl,Sawyer,Norm,...,Typ,NaN,CarPort,Unf,TA,TA,Y,NaN,WD,Normal
2,141000,RM,Pave,Reg,Bnk,AllPub,Inside,Gtl,IDOTRR,Norm,...,Typ,Gd,Detchd,Unf,TA,TA,P,NaN,WD,Normal
3,254000,FV,Pave,IR1,Lvl,AllPub,Inside,Gtl,Somerst,Norm,...,Typ,TA,Attchd,RFn,TA,TA,Y,NaN,WD,Normal
4,155000,RL,Pave,Reg,Lvl,AllPub,Inside,Gtl,SWISU,Feedr,...,Typ,NaN,Detchd,RFn,TA,TA,Y,MnPrv,WD,Normal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2632,114500,RM,Pave,Reg,Lvl,AllPub,Inside,Gtl,BrkSide,Norm,...,Typ,Gd,Detchd,Unf,TA,TA,Y,MnPrv,WD,Normal
2633,162000,RL,Pave,IR1,Lvl,AllPub,Inside,Gtl,SawyerW,Feedr,...,Typ,NaN,Attchd,Unf,TA,TA,Y,NaN,WD,Normal
2634,211500,RL,Pave,IR1,Lvl,AllPub,CulDSac,Gtl,Greens,Norm,...,Typ,TA,Attchd,Fin,TA,TA,Y,NaN,WD,Normal
2635,165000,RL,Pave,Reg,Lvl,AllPub,Inside,Gtl,NWAmes,Norm,...,Typ,TA,Attchd,RFn,TA,TA,Y,MnPrv,WD,Normal


4. Drop remaining records with null counts (if few)

In [1087]:
# backup 2
df_o_bak2 = df.copy()

In [1088]:
# restore from backup 2
df = df_o_bak2.copy()

In [1089]:
df

,SalePrice,MS Zoning,Street,Lot Shape,Land Contour,Utilities,Lot Config,Land Slope,Neighborhood,Condition 1,...,Functional,Fireplace Qu,Garage Type,Garage Finish,Garage Qual,Garage Cond,Paved Drive,Fence,Sale Type,Sale Condition
0,115400,RL,Pave,Reg,Lvl,AllPub,Inside,Gtl,Sawyer,Feedr,...,Typ,NaN,Detchd,Unf,TA,TA,N,MnPrv,WD,Normal
1,90000,RL,Pave,Reg,Lvl,AllPub,Inside,Gtl,Sawyer,Norm,...,Typ,NaN,CarPort,Unf,TA,TA,Y,NaN,WD,Normal
2,141000,RM,Pave,Reg,Bnk,AllPub,Inside,Gtl,IDOTRR,Norm,...,Typ,Gd,Detchd,Unf,TA,TA,P,NaN,WD,Normal
3,254000,FV,Pave,IR1,Lvl,AllPub,Inside,Gtl,Somerst,Norm,...,Typ,TA,Attchd,RFn,TA,TA,Y,NaN,WD,Normal
4,155000,RL,Pave,Reg,Lvl,AllPub,Inside,Gtl,SWISU,Feedr,...,Typ,NaN,Detchd,RFn,TA,TA,Y,MnPrv,WD,Normal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2632,114500,RM,Pave,Reg,Lvl,AllPub,Inside,Gtl,BrkSide,Norm,...,Typ,Gd,Detchd,Unf,TA,TA,Y,MnPrv,WD,Normal
2633,162000,RL,Pave,IR1,Lvl,AllPub,Inside,Gtl,SawyerW,Feedr,...,Typ,NaN,Attchd,Unf,TA,TA,Y,NaN,WD,Normal
2634,211500,RL,Pave,IR1,Lvl,AllPub,CulDSac,Gtl,Greens,Norm,...,Typ,TA,Attchd,Fin,TA,TA,Y,NaN,WD,Normal
2635,165000,RL,Pave,Reg,Lvl,AllPub,Inside,Gtl,NWAmes,Norm,...,Typ,TA,Attchd,RFn,TA,TA,Y,MnPrv,WD,Normal


In [1090]:
# Select columns where the missing value ratio is less than drop_column_threshold
low_nulls = df.columns[df.isnull().mean() < drop_row_threshold]
low_nulls

Index(['SalePrice', 'MS Zoning', 'Street', 'Lot Shape', 'Land Contour',
       'Utilities', 'Lot Config', 'Land Slope', 'Neighborhood', 'Condition 1',
       'Condition 2', 'Bldg Type', 'House Style', 'Roof Style', 'Roof Matl',
       'Exterior 1st', 'Exterior 2nd', 'Exter Qual', 'Exter Cond',
       'Foundation', 'Bsmt Qual', 'Bsmt Cond', 'Bsmt Exposure',
       'BsmtFin Type 1', 'BsmtFin Type 2', 'Heating', 'Heating QC',
       'Central Air', 'Electrical', 'Kitchen Qual', 'Functional',
       'Garage Type', 'Garage Finish', 'Garage Qual', 'Garage Cond',
       'Paved Drive', 'Sale Type', 'Sale Condition'],
      dtype='object')

In [1091]:
# Drop those specific columns
df = df.dropna(subset=low_nulls)
df

,SalePrice,MS Zoning,Street,Lot Shape,Land Contour,Utilities,Lot Config,Land Slope,Neighborhood,Condition 1,...,Functional,Fireplace Qu,Garage Type,Garage Finish,Garage Qual,Garage Cond,Paved Drive,Fence,Sale Type,Sale Condition
0,115400,RL,Pave,Reg,Lvl,AllPub,Inside,Gtl,Sawyer,Feedr,...,Typ,NaN,Detchd,Unf,TA,TA,N,MnPrv,WD,Normal
2,141000,RM,Pave,Reg,Bnk,AllPub,Inside,Gtl,IDOTRR,Norm,...,Typ,Gd,Detchd,Unf,TA,TA,P,NaN,WD,Normal
3,254000,FV,Pave,IR1,Lvl,AllPub,Inside,Gtl,Somerst,Norm,...,Typ,TA,Attchd,RFn,TA,TA,Y,NaN,WD,Normal
4,155000,RL,Pave,Reg,Lvl,AllPub,Inside,Gtl,SWISU,Feedr,...,Typ,NaN,Detchd,RFn,TA,TA,Y,MnPrv,WD,Normal
5,125000,RL,Pave,Reg,Lvl,AllPub,FR2,Gtl,Sawyer,Feedr,...,Typ,Fa,Detchd,Unf,TA,TA,Y,MnPrv,WD,Normal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2632,114500,RM,Pave,Reg,Lvl,AllPub,Inside,Gtl,BrkSide,Norm,...,Typ,Gd,Detchd,Unf,TA,TA,Y,MnPrv,WD,Normal
2633,162000,RL,Pave,IR1,Lvl,AllPub,Inside,Gtl,SawyerW,Feedr,...,Typ,NaN,Attchd,Unf,TA,TA,Y,NaN,WD,Normal
2634,211500,RL,Pave,IR1,Lvl,AllPub,CulDSac,Gtl,Greens,Norm,...,Typ,TA,Attchd,Fin,TA,TA,Y,NaN,WD,Normal
2635,165000,RL,Pave,Reg,Lvl,AllPub,Inside,Gtl,NWAmes,Norm,...,Typ,TA,Attchd,RFn,TA,TA,Y,MnPrv,WD,Normal


5.  Impute with the median for remaining fields

In [1092]:
# backup 3
df_o_bak3 = df.copy()

In [1093]:
# restore from backup 3
df = df_o_bak3.copy()

In [1094]:
# check again how many of each column is null
df.isnull().sum().sort_values( ascending = False )

,0
Fence,1937
Mas Vnr Type,1420
Fireplace Qu,1108
MS Zoning,0
Land Contour,0
Utilities,0
Street,0
Lot Shape,0
SalePrice,0
Neighborhood,0


Impute:

In [1095]:
# Get list of feature cols
feature_cols = list(df.drop(columns = target_col).columns)

# Loop through feature cols to bulk impute
for feature_col in feature_cols:

  # Assign mode to nulls
  mode = df[feature_col].mode(dropna = True)[0]
  df.fillna( {feature_col: mode}, inplace = True )

In [1096]:
# check again how many of each column is null
df.isnull().sum().sort_values( ascending = False )

,0
SalePrice,0
MS Zoning,0
Street,0
Lot Shape,0
Land Contour,0
Utilities,0
Lot Config,0
Land Slope,0
Neighborhood,0
Condition 1,0


In [1097]:
# check total null counts sum
f"Number of remaining nulls in df: {int(df.isnull().sum().sum())}"

'Number of remaining nulls in df: 0'

In [1098]:
# backup
df_object_bak = df_object.copy()

In [1099]:
# restore
df_object = df_object_bak.copy()

In [1100]:
# update
df_object = df.copy()

## Exploratory Data Analysis


### EDA — Floats

In [1101]:
# set df to df_float
df = df_float.copy()

### EDA — Ints

In [1102]:
# backup
df_float_bak2 = df_float.copy()

In [1103]:
# restore
df_float = df_float_bak2.copy()

In [1104]:
# update
df_float = df.copy()

In [1105]:
# set df to df_int
df = df_int.copy()

### EDA — Objects

In [1106]:
# backup
df_int_bak2 = df_int.copy()

In [1107]:
# restore
df_int = df_int_bak2.copy()

In [1108]:
# update
df_int = df.copy()

In [1109]:
# set df to df_object
df = df_object.copy()

#### Encoding — Objects (Categorical Data)

Next, we encode the object features in order to look at correlations:

In [1110]:
# Assign numerical values and store in another column
df_encoded = pd.DataFrame(df[target_col])
for feature in feature_cols:
  df_feature = pd.get_dummies(
      pd.DataFrame(df[feature]),
      columns=[feature],
      prefix=feature,
      prefix_sep="_",
      dtype="int",
      #drop_first=True
      )

  df_encoded = pd.concat([df_encoded, df_feature], axis=1, ignore_index=False)

df_encoded

,SalePrice,MS Zoning_C (all),MS Zoning_FV,MS Zoning_I (all),MS Zoning_RH,MS Zoning_RL,MS Zoning_RM,Street_Grvl,Street_Pave,Lot Shape_IR1,...,Sale Type_New,Sale Type_Oth,Sale Type_VWD,Sale Type_WD,Sale Condition_Abnorml,Sale Condition_AdjLand,Sale Condition_Alloca,Sale Condition_Family,Sale Condition_Normal,Sale Condition_Partial
0,115400,0,0,0,0,1,0,0,1,0,...,0,0,0,1,0,0,0,0,1,0
2,141000,0,0,0,0,0,1,0,1,0,...,0,0,0,1,0,0,0,0,1,0
3,254000,0,1,0,0,0,0,0,1,1,...,0,0,0,1,0,0,0,0,1,0
4,155000,0,0,0,0,1,0,0,1,0,...,0,0,0,1,0,0,0,0,1,0
5,125000,0,0,0,0,1,0,0,1,0,...,0,0,0,1,0,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2632,114500,0,0,0,0,0,1,0,1,0,...,0,0,0,1,0,0,0,0,1,0
2633,162000,0,0,0,0,1,0,0,1,1,...,0,0,0,1,0,0,0,0,1,0
2634,211500,0,0,0,0,1,0,0,1,1,...,0,0,0,1,0,0,0,0,1,0
2635,165000,0,0,0,0,1,0,0,1,0,...,0,0,0,1,0,0,0,0,1,0


In [1111]:
# backup 4
df_o_bak4 = df.copy()

In [1112]:
# restore from backup 4
df = df_o_bak4.copy()

In [1113]:
df

,SalePrice,MS Zoning,Street,Lot Shape,Land Contour,Utilities,Lot Config,Land Slope,Neighborhood,Condition 1,...,Functional,Fireplace Qu,Garage Type,Garage Finish,Garage Qual,Garage Cond,Paved Drive,Fence,Sale Type,Sale Condition
0,115400,RL,Pave,Reg,Lvl,AllPub,Inside,Gtl,Sawyer,Feedr,...,Typ,Gd,Detchd,Unf,TA,TA,N,MnPrv,WD,Normal
2,141000,RM,Pave,Reg,Bnk,AllPub,Inside,Gtl,IDOTRR,Norm,...,Typ,Gd,Detchd,Unf,TA,TA,P,MnPrv,WD,Normal
3,254000,FV,Pave,IR1,Lvl,AllPub,Inside,Gtl,Somerst,Norm,...,Typ,TA,Attchd,RFn,TA,TA,Y,MnPrv,WD,Normal
4,155000,RL,Pave,Reg,Lvl,AllPub,Inside,Gtl,SWISU,Feedr,...,Typ,Gd,Detchd,RFn,TA,TA,Y,MnPrv,WD,Normal
5,125000,RL,Pave,Reg,Lvl,AllPub,FR2,Gtl,Sawyer,Feedr,...,Typ,Fa,Detchd,Unf,TA,TA,Y,MnPrv,WD,Normal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2632,114500,RM,Pave,Reg,Lvl,AllPub,Inside,Gtl,BrkSide,Norm,...,Typ,Gd,Detchd,Unf,TA,TA,Y,MnPrv,WD,Normal
2633,162000,RL,Pave,IR1,Lvl,AllPub,Inside,Gtl,SawyerW,Feedr,...,Typ,Gd,Attchd,Unf,TA,TA,Y,MnPrv,WD,Normal
2634,211500,RL,Pave,IR1,Lvl,AllPub,CulDSac,Gtl,Greens,Norm,...,Typ,TA,Attchd,Fin,TA,TA,Y,MnPrv,WD,Normal
2635,165000,RL,Pave,Reg,Lvl,AllPub,Inside,Gtl,NWAmes,Norm,...,Typ,TA,Attchd,RFn,TA,TA,Y,MnPrv,WD,Normal


In [1114]:
# update df
df = df_encoded.copy()

## Processing



In [1115]:
# backup
df_object_bak2 = df_object.copy()

In [1116]:
# restore
df_object = df_object_bak2.copy()

In [1117]:
# update
df_object = df.copy()

### Merging Data

In [1118]:
# backup
df_all_bak = df_all.copy()

In [1119]:
# restore
df_all = df_all_bak.copy()

In [1120]:
df_all = pd.concat([
    df_all[[target_col]],
    df_float.drop(columns=[target_col]),
    df_int.drop(columns=[target_col]),
    df_object.drop(columns=[target_col])
    ],
    axis = 1)

Concat gives the equivalent of an outer join, meaning that for fields in one column where a row was dropped in another, we insert nulls.

In [1121]:
df_all

,SalePrice,Lot Frontage,Mas Vnr Area,BsmtFin SF 1,BsmtFin SF 2,Bsmt Unf SF,Total Bsmt SF,Bsmt Full Bath,Bsmt Half Bath,Garage Yr Blt,...,Sale Type_New,Sale Type_Oth,Sale Type_VWD,Sale Type_WD,Sale Condition_Abnorml,Sale Condition_AdjLand,Sale Condition_Alloca,Sale Condition_Family,Sale Condition_Normal,Sale Condition_Partial
0,115400,62.0,0.0,513.0,144.0,348.0,1005.0,1.0,0.0,1964.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
1,90000,72.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1967.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,141000,50.0,0.0,0.0,0.0,780.0,780.0,0.0,0.0,1979.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
3,254000,68.0,0.0,484.0,0.0,491.0,975.0,1.0,0.0,2000.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
4,155000,60.0,0.0,350.0,210.0,0.0,560.0,1.0,0.0,1971.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2632,114500,52.0,0.0,0.0,0.0,816.0,816.0,0.0,0.0,1985.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
2633,162000,74.0,0.0,460.0,286.0,308.0,1054.0,1.0,0.0,1990.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
2634,211500,61.0,0.0,960.0,0.0,266.0,1226.0,1.0,0.0,1981.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
2635,165000,78.0,174.0,0.0,0.0,1064.0,1064.0,0.0,0.0,1974.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0


In [1122]:
df_all.isna().sum().sort_values(ascending=False)

,0
Sale Condition_Partial,211
Sale Condition_Normal,211
Sale Condition_Family,211
Sale Condition_Alloca,211
Sale Condition_AdjLand,211
...,...
3Ssn Porch,0
Enclosed Porch,0
Yr Sold,0
Mo Sold,0


To achieve an inner join, we simply drop rows where nulls appeared during concat:

In [1123]:
df_all.dropna()

,SalePrice,Lot Frontage,Mas Vnr Area,BsmtFin SF 1,BsmtFin SF 2,Bsmt Unf SF,Total Bsmt SF,Bsmt Full Bath,Bsmt Half Bath,Garage Yr Blt,...,Sale Type_New,Sale Type_Oth,Sale Type_VWD,Sale Type_WD,Sale Condition_Abnorml,Sale Condition_AdjLand,Sale Condition_Alloca,Sale Condition_Family,Sale Condition_Normal,Sale Condition_Partial
0,115400,62.0,0.0,513.0,144.0,348.0,1005.0,1.0,0.0,1964.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
2,141000,50.0,0.0,0.0,0.0,780.0,780.0,0.0,0.0,1979.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
3,254000,68.0,0.0,484.0,0.0,491.0,975.0,1.0,0.0,2000.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
4,155000,60.0,0.0,350.0,210.0,0.0,560.0,1.0,0.0,1971.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
5,125000,70.0,0.0,842.0,0.0,70.0,912.0,0.0,0.0,1974.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2632,114500,52.0,0.0,0.0,0.0,816.0,816.0,0.0,0.0,1985.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
2633,162000,74.0,0.0,460.0,286.0,308.0,1054.0,1.0,0.0,1990.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
2634,211500,61.0,0.0,960.0,0.0,266.0,1226.0,1.0,0.0,1981.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
2635,165000,78.0,174.0,0.0,0.0,1064.0,1064.0,0.0,0.0,1974.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0


In [1124]:
# backup
df_all_bak2 = df_all.copy()

In [1125]:
# restore
df_all = df_all_bak2.copy()

In [1126]:
df_all = df_all.dropna()

In [1128]:
df = df_all.copy()

In [1129]:
df

,SalePrice,Lot Frontage,Mas Vnr Area,BsmtFin SF 1,BsmtFin SF 2,Bsmt Unf SF,Total Bsmt SF,Bsmt Full Bath,Bsmt Half Bath,Garage Yr Blt,...,Sale Type_New,Sale Type_Oth,Sale Type_VWD,Sale Type_WD,Sale Condition_Abnorml,Sale Condition_AdjLand,Sale Condition_Alloca,Sale Condition_Family,Sale Condition_Normal,Sale Condition_Partial
0,115400,62.0,0.0,513.0,144.0,348.0,1005.0,1.0,0.0,1964.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
2,141000,50.0,0.0,0.0,0.0,780.0,780.0,0.0,0.0,1979.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
3,254000,68.0,0.0,484.0,0.0,491.0,975.0,1.0,0.0,2000.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
4,155000,60.0,0.0,350.0,210.0,0.0,560.0,1.0,0.0,1971.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
5,125000,70.0,0.0,842.0,0.0,70.0,912.0,0.0,0.0,1974.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2632,114500,52.0,0.0,0.0,0.0,816.0,816.0,0.0,0.0,1985.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
2633,162000,74.0,0.0,460.0,286.0,308.0,1054.0,1.0,0.0,1990.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
2634,211500,61.0,0.0,960.0,0.0,266.0,1226.0,1.0,0.0,1981.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
2635,165000,78.0,174.0,0.0,0.0,1064.0,1064.0,0.0,0.0,1974.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0


## Convert CSV to parquet


In [1130]:
csv_file = "housing.csv"
csv_file


'housing.csv'

In [1131]:
parquet_file = "housing.parquet"
parquet_file


'housing.parquet'

In [1132]:
df.to_parquet( parquet_file, index=False)

In [1133]:
df.to_csv( csv_file, index=False)

In [1134]:
! ls -l --si {parquet_file} {csv_file}


-rw-r--r-- 1 root root 2.8M Nov  3 12:58 housing.csv
-rw-r--r-- 1 root root 333k Nov  3 12:58 housing.parquet


## Verify that pandas can query the local parquet file


In [1135]:
df2 = pd.read_parquet( parquet_file )
df2.shape


(2404, 285)

In [1136]:
df3 = pd.read_csv( csv_file )
df3.shape

(2404, 285)

## Authenticate to Hugging Face


In [1137]:
from google.colab import userdata
import os

In [1138]:
# Retrieve the token from Colab's secret manager
os.environ["HF_TOKEN"] = userdata.get('hf_token')
_ = os.environ["HF_TOKEN"]
f"{_[:3]} ... {_[-4:]}"

'hf_ ... Tuux'

In [1139]:
os.environ["HF_ACCOUNT"] = userdata.get('hf_account')
hf_account = os.environ["HF_ACCOUNT"]
hf_account


'timnickellddds'

In [1140]:
hf_repo = "Data_Science-22"
os.environ["HF_REPO"] = hf_repo
hf_repo


'Data_Science-22'

In [1141]:
# Log in automatically without an interactive prompt
! hf auth login --token $HF_TOKEN


Hint: A new version of huggingface_hub (2.4.0) is available! You are using version 1.33.0.
To update, run: hf update
Hint: The `hf-cli` skill is not installed. Run `hf skills add -g` to teach your AI agents how to use the `hf` CLI.
The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: fineGrained).
The token `Colab` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## Push to Hugging Face


In [1144]:
%%capture hf_upload
%%bash

hf upload \
  --type dataset \
  ${HF_ACCOUNT}/${HF_REPO} \
  housing.parquet


In [1145]:
print(hf_upload.stdout)


✓ Uploaded
  url: https://huggingface.co/datasets/timnickellddds/Data_Science-22/commit/b2ec39d12bcc4ce1037722ba0c42ace99787d59f



In [1146]:
%%capture hf_upload
%%bash

hf upload \
  --type dataset \
  ${HF_ACCOUNT}/${HF_REPO} \
  housing.csv


In [1147]:
print(hf_upload.stdout)


✓ Uploaded
  url: https://huggingface.co/datasets/timnickellddds/Data_Science-22/commit/4697421482fe6c3a9f7957427528e5a2f7f0ccc4



## Read parquet from Hugging Face


In [1148]:
# Construct the base Hugging Face URL for your dataset files
hf_url = f"https://huggingface.co/datasets/{hf_account}/{hf_repo}/resolve/main/housing.parquet"
hf_url


'https://huggingface.co/datasets/timnickellddds/Data_Science-22/resolve/main/housing.parquet'

In [1149]:
df3 = pd.read_parquet( hf_url )
df3.shape


(2404, 285)

In [1150]:
df3.iloc[:,:5].info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2404 entries, 0 to 2403
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   SalePrice     2404 non-null   int64  
 1   Lot Frontage  2404 non-null   float64
 2   Mas Vnr Area  2404 non-null   float64
 3   BsmtFin SF 1  2404 non-null   float64
 4   BsmtFin SF 2  2404 non-null   float64
dtypes: float64(4), int64(1)
memory usage: 94.0 KB


## Read CSV from Hugging Face


In [1151]:
# Construct the base Hugging Face URL for your dataset files
hf_url = f"https://huggingface.co/datasets/{hf_account}/{hf_repo}/resolve/main/housing.csv"
hf_url


'https://huggingface.co/datasets/timnickellddds/Data_Science-22/resolve/main/housing.csv'

In [1152]:
df4 = pd.read_csv( hf_url )
df4.shape


(2404, 285)

In [1153]:
df4.iloc[:,:5].info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2404 entries, 0 to 2403
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   SalePrice     2404 non-null   int64  
 1   Lot Frontage  2404 non-null   float64
 2   Mas Vnr Area  2404 non-null   float64
 3   BsmtFin SF 1  2404 non-null   float64
 4   BsmtFin SF 2  2404 non-null   float64
dtypes: float64(4), int64(1)
memory usage: 94.0 KB
